Управляющий ноутбук будет:
- последовательно обрабатывать структуры, по одной за раз;
- проверять результаты каждого этапа;
- пропускать уже успешно обработанные структуры;
- повторять обработку неудачных структур;
- вести общий журнал ошибок и времени выполнения;
- собирать сводные таблицы по всем белкам и найденным карманам.
Важный принцип: сохраняем все обнаруженные карманы, а не только лучший. Отбор наиболее перспективных карманов будет отдельным этапом анализа.

In [1]:
# ============================================================
# CELL 1. ENVIRONMENT AND IMPORTS
# ============================================================
# Назначение:
# 1. Подготовка среды пакетной обработки.
# 2. Импорт библиотек.
# 3. Проверка необходимых модулей.
# 4. Настройка путей проекта.
# ============================================================

import os
import sys
import re
import json
import time
import shutil
import traceback
import warnings

from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd

try:
    import nbformat
    from nbclient import NotebookClient
    from tqdm.auto import tqdm

except ImportError as exc:
    raise ImportError(
        "Не установлены необходимые библиотеки.\n"
        "Выполните один раз в отдельной ячейке:\n"
        "%pip install nbformat nbclient tqdm"
    ) from exc


# ============================================================
# PROJECT PATHS
# ============================================================

PROJECT_ROOT = Path(
    r"C:\apex\APEX\МИФИ\СНО\medic"
)

DATASET_ROOT = (
    PROJECT_ROOT / "Protein_Pocket_Dataset"
)

STRUCTURE_ANALYSIS_DIR = (
    DATASET_ROOT / "07_structure_analysis"
)

STRUCTURES_DIR = (
    STRUCTURE_ANALYSIS_DIR / "structures"
)

RESULTS_ROOT = (
    STRUCTURE_ANALYSIS_DIR / "results"
)

SELECTION_CSV = (
    DATASET_ROOT
    / "05_final_dataset"
    / "protein_final_100.csv"
)

# Локальный путь к исходному ноутбуку.
ANALYZER_NOTEBOOK_PATH = (
    PROJECT_ROOT / "Protein Structure Analyzer.ipynb"
)

# Каталог пакетного управления.
BATCH_ROOT = (
    STRUCTURE_ANALYSIS_DIR / "batch_runs"
)

BATCH_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

print("=" * 70)
print("BATCH PROTEIN PROCESSOR")
print("=" * 70)

print("Python:", sys.version)
print("Project root:", PROJECT_ROOT)
print("Analyzer notebook:", ANALYZER_NOTEBOOK_PATH)
print("Selection CSV:", SELECTION_CSV)
print("Results root:", RESULTS_ROOT)

print("=" * 70)

# Проверки путей.

if not PROJECT_ROOT.exists():
    raise FileNotFoundError(
        f"PROJECT_ROOT не существует:\n{PROJECT_ROOT}"
    )

if not SELECTION_CSV.is_file():
    raise FileNotFoundError(
        f"Не найден CSV отбора:\n{SELECTION_CSV}"
    )

if not ANALYZER_NOTEBOOK_PATH.is_file():
    raise FileNotFoundError(
        f"Не найден исходный ноутбук:\n"
        f"{ANALYZER_NOTEBOOK_PATH}\n\n"
        "Исправьте ANALYZER_NOTEBOOK_PATH "
        "на фактический локальный путь."
    )

print("Environment validation: OK")

BATCH PROTEIN PROCESSOR
Python: 3.10.6 (tags/v3.10.6:9c7b4bd, Aug  1 2022, 21:53:49) [MSC v.1932 64 bit (AMD64)]
Project root: C:\apex\APEX\МИФИ\СНО\medic
Analyzer notebook: C:\apex\APEX\МИФИ\СНО\medic\Protein Structure Analyzer.ipynb
Selection CSV: C:\apex\APEX\МИФИ\СНО\medic\Protein_Pocket_Dataset\05_final_dataset\protein_final_100.csv
Results root: C:\apex\APEX\МИФИ\СНО\medic\Protein_Pocket_Dataset\07_structure_analysis\results
Environment validation: OK


In [2]:

# ============================================================
# CELL 2. BATCH CONFIGURATION
# ============================================================
# Назначение:
# 1. Выбор режима пакетной обработки.
# 2. Настройка количества PDB.
# 3. Настройка возобновления.
# 4. Настройка времени выполнения.
# 5. Автоматическое определение Jupyter Kernel.
# 6. Настройка целевых компонентов по PDB.
# ============================================================


# ------------------------------------------------------------
# 1. PROCESSING MODE
# ------------------------------------------------------------

# TEST  — первые N структур.
# BATCH — весь список или выбранное подмножество.

BATCH_MODE = "BATCH"

ALLOWED_BATCH_MODES = {
    "TEST",
    "BATCH"
}

if BATCH_MODE not in ALLOWED_BATCH_MODES:
    raise ValueError(
        f"Недопустимый BATCH_MODE: {BATCH_MODE}"
    )


# ------------------------------------------------------------
# 2. PDB SELECTION
# ------------------------------------------------------------

TEST_SIZE = 5

# None — использовать весь список из CSV.
# Или указать конкретные PDB:
# CUSTOM_PDB_IDS = ["1OCC", "1ABC"]

CUSTOM_PDB_IDS = None

# Начальный индекс в исходном списке.
BATCH_START_INDEX = 0

# Максимальное количество структур.
# None — без ограничения.

BATCH_LIMIT = None


# ------------------------------------------------------------
# 3. RESUME SETTINGS
# ------------------------------------------------------------

# Пропускать ранее успешно обработанные PDB,
# если итоговые файлы существуют.

SKIP_COMPLETED = True

# Повторять структуры, ранее завершившиеся ошибкой.

RETRY_FAILED = True


# ------------------------------------------------------------
# 4. EXECUTION SETTINGS
# ------------------------------------------------------------

# Максимальное время выполнения одной ячейки
# исходного ноутбука, секунды.

CELL_TIMEOUT = 3600

# Имя ядра определяется автоматически
# из metadata исходного ноутбука.

KERNEL_NAME = None

if not ANALYZER_NOTEBOOK_PATH.exists():
    raise FileNotFoundError(
        "Исходный ноутбук не найден:\n"
        f"{ANALYZER_NOTEBOOK_PATH}"
    )

# Чтение metadata исходного ноутбука.

import nbformat

analyzer_metadata = nbformat.read(
    ANALYZER_NOTEBOOK_PATH,
    as_version=4
).metadata

notebook_kernel = (
    analyzer_metadata
    .get("kernelspec", {})
    .get("name")
)

if KERNEL_NAME is None:
    KERNEL_NAME = notebook_kernel

if not isinstance(KERNEL_NAME, str) or not KERNEL_NAME.strip():
    raise ValueError(
        "Не удалось определить Jupyter Kernel.\n"
        "Проверь metadata исходного ноутбука "
        "или укажи KERNEL_NAME вручную."
    )

KERNEL_NAME = KERNEL_NAME.strip()

# Сохранять исполненные копии исходного ноутбука.
# Для 100 структур может занимать много места.

SAVE_EXECUTED_NOTEBOOK = False


# ------------------------------------------------------------
# 5. BATCH RUN IDENTIFIER
# ------------------------------------------------------------

# Имя должно оставаться неизменным,
# если необходимо продолжить прежний запуск.

BATCH_RUN_NAME = "protein_batch_v1"

BATCH_RUN_DIR = (
    BATCH_ROOT / BATCH_RUN_NAME
)

BATCH_RUN_DIR.mkdir(
    parents=True,
    exist_ok=True
)

BATCH_LOG_DIR = (
    BATCH_RUN_DIR / "logs"
)

BATCH_NOTEBOOK_DIR = (
    BATCH_RUN_DIR / "executed_notebooks"
)

BATCH_LOG_DIR.mkdir(
    parents=True,
    exist_ok=True
)

if SAVE_EXECUTED_NOTEBOOK:
    BATCH_NOTEBOOK_DIR.mkdir(
        parents=True,
        exist_ok=True
    )


# ------------------------------------------------------------
# 6. COMPONENT SELECTION REGISTRY
# ------------------------------------------------------------

# Выбор целевого небелкового компонента задаётся
# индивидуально для каждого PDB.
#
# Не следует автоматически назначать HEM всем белкам.
#
# Пример:
#
# COMPONENT_SELECTION_BY_PDB = {
#     "1OCC": {
#         "component_name": "HEA",
#         "chain": "A",
#         "residue_number": None,
#         "component_index": None
#     }
# }
#
# Если PDB отсутствует в словаре:
# автоматический выбор компонента отключается.
#
# Это не мешает остальному анализу структуры.

COMPONENT_SELECTION_BY_PDB = {}


# ------------------------------------------------------------
# 7. CONFIGURATION VALIDATION
# ------------------------------------------------------------

if not isinstance(TEST_SIZE, int) or TEST_SIZE < 1:
    raise ValueError("TEST_SIZE должен быть целым числом >= 1.")

if not isinstance(BATCH_START_INDEX, int) or BATCH_START_INDEX < 0:
    raise ValueError("BATCH_START_INDEX должен быть целым числом >= 0.")

if BATCH_LIMIT is not None:
    if not isinstance(BATCH_LIMIT, int) or BATCH_LIMIT < 1:
        raise ValueError("BATCH_LIMIT должен быть None или целым числом >= 1.")

if CELL_TIMEOUT is not None:
    if not isinstance(CELL_TIMEOUT, int) or CELL_TIMEOUT < 1:
        raise ValueError("CELL_TIMEOUT должен быть положительным числом.")

if not isinstance(BATCH_RUN_NAME, str) or not BATCH_RUN_NAME.strip():
    raise ValueError("BATCH_RUN_NAME не может быть пустым.")

if not isinstance(COMPONENT_SELECTION_BY_PDB, dict):
    raise TypeError("COMPONENT_SELECTION_BY_PDB должен быть словарём.")


# ------------------------------------------------------------
# 8. CONFIGURATION SUMMARY
# ------------------------------------------------------------

print("=" * 70)
print("BATCH CONFIGURATION")
print("=" * 70)

print("Mode:", BATCH_MODE)
print("Test size:", TEST_SIZE)
print("Custom PDB list:", CUSTOM_PDB_IDS)
print("Start index:", BATCH_START_INDEX)
print("Batch limit:", BATCH_LIMIT)

print("-" * 70)

print("Skip completed:", SKIP_COMPLETED)
print("Retry failed:", RETRY_FAILED)
print("Cell timeout:", CELL_TIMEOUT)
print("Kernel:", KERNEL_NAME)

print("-" * 70)

print("Analyzer notebook:", ANALYZER_NOTEBOOK_PATH)
print("Kernel metadata:", notebook_kernel)

print("-" * 70)

print("Component registry entries:",
      len(COMPONENT_SELECTION_BY_PDB))

print("Batch run:", BATCH_RUN_NAME)
print("Batch directory:", BATCH_RUN_DIR)

print("-" * 70)

print("Environment validation: OK")
print("=" * 70)


BATCH CONFIGURATION
Mode: BATCH
Test size: 5
Custom PDB list: None
Start index: 0
Batch limit: None
----------------------------------------------------------------------
Skip completed: True
Retry failed: True
Cell timeout: 3600
Kernel: python3
----------------------------------------------------------------------
Analyzer notebook: C:\apex\APEX\МИФИ\СНО\medic\Protein Structure Analyzer.ipynb
Kernel metadata: python3
----------------------------------------------------------------------
Component registry entries: 0
Batch run: protein_batch_v1
Batch directory: C:\apex\APEX\МИФИ\СНО\medic\Protein_Pocket_Dataset\07_structure_analysis\batch_runs\protein_batch_v1
----------------------------------------------------------------------
Environment validation: OK


In [3]:

# ============================================================
# CELL 3. PDB REGISTRY AND PROCESSING STATE
# ============================================================
# Назначение:
# 1. Загрузка списка PDB.
# 2. Проверка идентификаторов.
# 3. Формирование списка обработки.
# 4. Загрузка предыдущих статусов.
# 5. Подготовка реестра результатов.
# ============================================================


# ------------------------------------------------------------
# 1. LOAD PDB REGISTRY
# ------------------------------------------------------------

if not SELECTION_CSV.is_file():
    raise FileNotFoundError(
        f"Selection CSV не найден:\n{SELECTION_CSV}"
    )

selection_df = pd.read_csv(
    SELECTION_CSV,
    dtype={"pdb_id": "string"}
)

if "pdb_id" not in selection_df.columns:
    raise ValueError(
        "В CSV отсутствует столбец pdb_id."
    )

selection_df["pdb_id"] = (
    selection_df["pdb_id"]
    .astype("string")
    .str.strip()
    .str.upper()
)

selection_df = selection_df.dropna(
    subset=["pdb_id"]
).copy()

selection_df = selection_df[
    selection_df["pdb_id"] != ""
].copy()

if selection_df.empty:
    raise ValueError(
        "После очистки CSV не осталось PDB ID."
    )


# ------------------------------------------------------------
# 2. SORT REGISTRY
# ------------------------------------------------------------

if "selection_order" in selection_df.columns:

    selection_df["selection_order"] = pd.to_numeric(
        selection_df["selection_order"],
        errors="coerce"
    )

    selection_df = selection_df.sort_values(
        "selection_order",
        na_position="last"
    )

selection_df = selection_df.reset_index(
    drop=True
)


# ------------------------------------------------------------
# 3. VALIDATE PDB IDS
# ------------------------------------------------------------

invalid_ids = [
    pdb_id
    for pdb_id in selection_df["pdb_id"].astype(str)
    if not re.fullmatch(r"[A-Z0-9]{4}", pdb_id)
]

if invalid_ids:
    raise ValueError(
        f"Некорректные PDB ID: {invalid_ids[:10]}"
    )

if selection_df["pdb_id"].duplicated().any():

    duplicates = (
        selection_df.loc[
            selection_df["pdb_id"].duplicated(keep=False),
            "pdb_id"
        ]
        .tolist()
    )

    raise ValueError(
        f"Повторяющиеся PDB ID: {duplicates[:10]}"
    )


# ------------------------------------------------------------
# 4. BUILD PROCESSING LIST
# ------------------------------------------------------------

AVAILABLE_PDB_IDS = (
    selection_df["pdb_id"]
    .astype(str)
    .tolist()
)

if CUSTOM_PDB_IDS is not None:

    PDB_IDS = [
        str(x).strip().upper()
        for x in CUSTOM_PDB_IDS
    ]

    if len(PDB_IDS) != len(set(PDB_IDS)):
        raise ValueError(
            "CUSTOM_PDB_IDS содержит повторяющиеся PDB ID."
        )

    invalid_custom_ids = [
        x for x in PDB_IDS
        if not re.fullmatch(r"[A-Z0-9]{4}", x)
    ]

    if invalid_custom_ids:
        raise ValueError(
            f"Некорректные CUSTOM PDB ID: {invalid_custom_ids}"
        )

    unknown_ids = [
        x for x in PDB_IDS
        if x not in AVAILABLE_PDB_IDS
    ]

    if unknown_ids:
        raise ValueError(
            "PDB отсутствуют в selection CSV: "
            f"{unknown_ids}"
        )

else:

    PDB_IDS = AVAILABLE_PDB_IDS.copy()


# ------------------------------------------------------------
# 5. APPLY PROCESSING LIMITS
# ------------------------------------------------------------

if not isinstance(BATCH_START_INDEX, int):
    raise TypeError(
        "BATCH_START_INDEX должен быть целым числом."
    )

if BATCH_START_INDEX < 0:
    raise ValueError(
        "BATCH_START_INDEX не может быть отрицательным."
    )

PDB_IDS = PDB_IDS[BATCH_START_INDEX:]


if BATCH_MODE == "TEST":
    PDB_IDS = PDB_IDS[:TEST_SIZE]


if BATCH_LIMIT is not None:
    PDB_IDS = PDB_IDS[:BATCH_LIMIT]


if not PDB_IDS:
    raise ValueError(
        "Список PDB для обработки пуст."
    )


# ------------------------------------------------------------
# 6. PROCESSING STATE FILES
# ------------------------------------------------------------

STATE_CSV = (
    BATCH_RUN_DIR / "batch_status.csv"
)

SUMMARY_CSV = (
    BATCH_RUN_DIR / "batch_summary.csv"
)

MANIFEST_JSON = (
    BATCH_RUN_DIR / "batch_manifest.json"
)


# ------------------------------------------------------------
# 7. LOAD PREVIOUS STATE
# ------------------------------------------------------------

STATE_COLUMNS = [
    "pdb_id",
    "status",
    "started_at",
    "finished_at",
    "duration_seconds",
    "output_dir",
    "error",
    "log_file"
]

if STATE_CSV.is_file():

    try:
        previous_state = pd.read_csv(
            STATE_CSV,
            dtype={"pdb_id": "string"}
        )

    except Exception as exc:
        raise RuntimeError(
            f"Не удалось прочитать файл состояния:\n"
            f"{STATE_CSV}\n"
            f"Причина: {exc}"
        ) from exc

    if "pdb_id" not in previous_state.columns:
        raise ValueError(
            "В batch_status.csv отсутствует столбец pdb_id."
        )

    previous_state["pdb_id"] = (
        previous_state["pdb_id"]
        .astype("string")
        .str.strip()
        .str.upper()
    )

else:

    previous_state = pd.DataFrame(
        columns=STATE_COLUMNS
    )


# ------------------------------------------------------------
# 8. NORMALIZE PREVIOUS STATE
# ------------------------------------------------------------

previous_state = previous_state.dropna(
    subset=["pdb_id"]
).copy()

previous_state = previous_state[
    previous_state["pdb_id"] != ""
].copy()

# Последняя запись по каждому PDB.

if not previous_state.empty:

    previous_state = (
        previous_state
        .drop_duplicates(
            subset=["pdb_id"],
            keep="last"
        )
        .reset_index(drop=True)
    )


PREVIOUS_STATUS = {
    str(row["pdb_id"]): row.to_dict()
    for _, row in previous_state.iterrows()
}


# ------------------------------------------------------------
# 9. DISPLAY REGISTRY
# ------------------------------------------------------------

print("=" * 70)
print("PDB REGISTRY")
print("=" * 70)

print("Total available:", len(AVAILABLE_PDB_IDS))
print("Selected for processing:", len(PDB_IDS))

print("-" * 70)

for i, pdb_id in enumerate(PDB_IDS, start=1):
    print(f"{i:3d}. {pdb_id}")

print("-" * 70)

print("Previous records:", len(PREVIOUS_STATUS))

print("=" * 70)


PDB REGISTRY
Total available: 100
Selected for processing: 100
----------------------------------------------------------------------
  1. 7WTU
  2. 7W37
  3. 7Q0G
  4. 7VYS
  5. 7X8R
  6. 7QB5
  7. 7Z0O
  8. 7XCR
  9. 12TB
 10. 3PUW
 11. 7WTT
 12. 3O2R
 13. 3NV0
 14. 7Z16
 15. 7WSO
 16. 7PR4
 17. 7W38
 18. 3OFG
 19. 7XNX
 20. 3P1A
 21. 7W1Z
 22. 7OYK
 23. 7WIG
 24. 7PS6
 25. 7W2U
 26. 3NZJ
 27. 7XCT
 28. 7YMS
 29. 3NY7
 30. 3P0Y
 31. 7Y3G
 32. 7YE4
 33. 7P4R
 34. 7W5Z
 35. 7PCA
 36. 7WU2
 37. 3OS8
 38. 7YE6
 39. 3P3O
 40. 7Z13
 41. 7PEF
 42. 7XD0
 43. 7PU4
 44. 3OR7
 45. 7W2Y
 46. 3O0A
 47. 7XMT
 48. 3PC8
 49. 21ZV
 50. 3O1C
 51. 7Y7M
 52. 3PSN
 53. 7W4C
 54. 3NWS
 55. 7Z38
 56. 3POZ
 57. 7VZ4
 58. 3OZZ
 59. 7Y75
 60. 3P8V
 61. 7XCP
 62. 3O1X
 63. 7XW6
 64. 3NZL
 65. 7W4F
 66. 3OSX
 67. 21ZW
 68. 7Q8Z
 69. 7Q2T
 70. 7Y76
 71. 3P8Y
 72. 7X3W
 73. 3OCM
 74. 7WZ4
 75. 7OX3
 76. 7W9K
 77. 7OYI
 78. 7XSZ
 79. 3O2P
 80. 3P6E
 81. 7X6O
 82. 7P8R
 83. 7XOV
 84. 7YI0
 85. 3NUL
 86. 3OM1
 87. 7

In [4]:

# ============================================================
# CELL 4. EXECUTION FUNCTIONS
# ============================================================
# Назначение:
# 1. Подготовка исходного ноутбука.
# 2. Передача PDB_ID в CELL 2.
# 3. Индивидуальная настройка CELL 17.
# 4. Исключение повторной установки библиотек.
# 5. Определение Jupyter Kernel.
# 6. Запуск NotebookClient.
# 7. Проверка результатов и их актуальности.
# 8. Сохранение журнала выполнения.
# 9. Защита от ошибок сохранения вспомогательных файлов.
# 10. Поддержка проверки старых результатов без
#     обязательного повторного обновления файлов.
# ============================================================


# ------------------------------------------------------------
# 0. IMPORTS
# ------------------------------------------------------------

import re
import json
import time
import traceback

from datetime import datetime

import numpy as np
import pandas as pd
import nbformat

from nbclient import NotebookClient


# ------------------------------------------------------------
# 1. NOTEBOOK OUTPUT EXTRACTION
# ------------------------------------------------------------

def extract_notebook_output(notebook):

    output_parts = []

    for cell_index, cell in enumerate(notebook.cells):

        if cell.cell_type != "code":
            continue

        for output in cell.get("outputs", []):

            output_type = output.get("output_type")

            if output_type == "stream":

                output_parts.append(
                    str(output.get("text", ""))
                )

            elif output_type == "error":

                output_parts.append(
                    "\n".join(
                        output.get("traceback", [])
                    )
                )

            elif output_type in {
                "display_data",
                "execute_result"
            }:

                data = output.get("data", {})

                text_value = data.get(
                    "text/plain",
                    ""
                )

                if text_value:

                    output_parts.append(
                        str(text_value)
                    )

    return "\n".join(output_parts)


# ------------------------------------------------------------
# 2. SKIP INSTALLATION COMMANDS
# ------------------------------------------------------------

def skip_installation_cells(notebook):

    skipped = 0

    installation_patterns = (
        "%pip install",
        "!pip install",
        "%conda install",
        "!conda install"
    )

    for cell in notebook.cells:

        if cell.cell_type != "code":
            continue

        original_lines = cell.source.splitlines()

        filtered_lines = []

        cell_changed = False

        for line in original_lines:

            stripped = line.strip().lower()

            is_installation = any(
                stripped.startswith(pattern)
                for pattern in installation_patterns
            )

            if is_installation:

                filtered_lines.append(
                    "# Installation command skipped by Batch Processor."
                )

                cell_changed = True

            else:

                filtered_lines.append(line)

        if cell_changed:

            cell.source = "\n".join(filtered_lines)

            skipped += 1

    return skipped


# ------------------------------------------------------------
# 3. PATCH COMPONENT SELECTION
# ------------------------------------------------------------

def patch_component_selection(
    notebook,
    pdb_id
):

    config = COMPONENT_SELECTION_BY_PDB.get(
        pdb_id,
        {}
    )

    target_cells = []

    for cell in notebook.cells:

        if cell.cell_type != "code":
            continue

        source = cell.source

        if (
            "PREFERRED_COMPONENT_NAME =" in source
            and "SELECT_COMPONENT_INDEX =" in source
        ):

            target_cells.append(cell)

    if len(target_cells) != 1:

        raise RuntimeError(
            "Не удалось однозначно определить CELL 17.\n"
            f"Найдено подходящих ячеек: {len(target_cells)}.\n"
            "Проверь настройки выбора компонента "
            "в исходном ноутбуке."
        )

    target_cell = target_cells[0]

    replacements = {

        "PREFERRED_COMPONENT_NAME":
            repr(config.get("component_name")),

        "PREFERRED_COMPONENT_CHAIN":
            repr(config.get("chain")),

        "PREFERRED_COMPONENT_RESIDUE_NUMBER":
            repr(config.get("residue_number")),

        "SELECT_COMPONENT_INDEX":
            repr(config.get("component_index"))
    }

    source = target_cell.source

    for variable_name, value in replacements.items():

        pattern = (
            rf"(?m)^([ \t]*){re.escape(variable_name)}"
            rf"\s*=\s*.*$"
        )

        source, count = re.subn(
            pattern,
            lambda match: (
                match.group(1)
                + variable_name
                + " = "
                + value
            ),
            source
        )

        if count != 1:

            raise RuntimeError(
                f"Не удалось заменить {variable_name} "
                f"в настройках CELL 17.\n"
                f"Количество совпадений: {count}.\n"
                "Проверь актуальный код CELL 17."
            )

    target_cell.source = source

    return config


# ------------------------------------------------------------
# 4. RESOLVE NOTEBOOK KERNEL
# ------------------------------------------------------------

def resolve_notebook_kernel(notebook):

    kernel_name = KERNEL_NAME

    if not kernel_name:

        kernel_name = (
            notebook.metadata
            .get("kernelspec", {})
            .get("name")
        )

    if not isinstance(kernel_name, str):

        raise RuntimeError(
            "Не удалось определить Jupyter Kernel.\n"
            "Проверь KERNEL_NAME и metadata исходного ноутбука."
        )

    kernel_name = kernel_name.strip()

    if not kernel_name:

        raise RuntimeError(
            "Имя Jupyter Kernel пустое."
        )

    return kernel_name


# ------------------------------------------------------------
# 5. PREPARE NOTEBOOK FOR ONE PDB
# ------------------------------------------------------------

def prepare_notebook_for_pdb(pdb_id):

    if not ANALYZER_NOTEBOOK_PATH.is_file():

        raise FileNotFoundError(
            f"Исходный ноутбук не найден:\n"
            f"{ANALYZER_NOTEBOOK_PATH}"
        )

    notebook = nbformat.read(
        ANALYZER_NOTEBOOK_PATH,
        as_version=4
    )

    # Проверяем ядро до запуска.

    notebook_kernel = resolve_notebook_kernel(
        notebook
    )

    # Удаляем только команды установки.

    skipped_install_cells = (
        skip_installation_cells(notebook)
    )

    # Настраиваем целевой компонент.

    component_config = patch_component_selection(
        notebook,
        pdb_id
    )

    # Передаём параметры до выполнения CELL 2.

    parameter_source = f"""
# ============================================================
# BATCH PROCESSOR PARAMETERS
# ============================================================

PDB_ID_INPUT = {pdb_id!r}

PDB_ID_OVERRIDE = None

BATCH_COMPONENT_SELECTION = {component_config!r}

print("Batch PDB ID:", PDB_ID_INPUT)
print("Component configuration:", BATCH_COMPONENT_SELECTION)
print("Notebook kernel:", {notebook_kernel!r})
"""

    parameter_cell = nbformat.v4.new_code_cell(
        source=parameter_source
    )

    parameter_cell.metadata["tags"] = [
        "batch_parameters"
    ]

    notebook.cells.insert(
        0,
        parameter_cell
    )

    return (
        notebook,
        skipped_install_cells,
        notebook_kernel
    )


# ------------------------------------------------------------
# 6. OUTPUT FILE SIGNATURE
# ------------------------------------------------------------

def get_file_signature(path):

    if path is None:

        return None

    if not path.is_file():

        return None

    stat = path.stat()

    return (
        stat.st_size,
        stat.st_mtime_ns
    )


# ------------------------------------------------------------
# 7. REQUIRED OUTPUT VALIDATION
# ------------------------------------------------------------

def validate_pdb_outputs(
    pdb_id,
    previous_signatures=None,
    require_fresh=True
):

    pdb_output_dir = RESULTS_ROOT / pdb_id

    if not pdb_output_dir.exists():

        return {

            "valid": False,

            "output_dir": str(pdb_output_dir),

            "graph_file": None,

            "features_json": None,

            "graph_valid": False,

            "features_valid": False,

            "graph_fresh": False,

            "features_fresh": False,

            "graph_candidates_count": 0,

            "features_candidates_count": 0,

            "message": "PDB results directory not found"
        }

    # --------------------------------------------------------
    # Поиск обязательных файлов
    # --------------------------------------------------------

    graph_candidates = sorted(
        pdb_output_dir.rglob(
            f"{pdb_id}_ml_graph_data.npz"
        )
    )

    feature_candidates = sorted(
        pdb_output_dir.rglob(
            f"{pdb_id}_protein_features.json"
        )
    )

    # При неоднозначности файл не выбирается.

    graph_file = (
        graph_candidates[0]
        if len(graph_candidates) == 1
        else None
    )

    features_json = (
        feature_candidates[0]
        if len(feature_candidates) == 1
        else None
    )

    graph_valid = False

    features_valid = False

    graph_fresh = False

    features_fresh = False

    graph_error = ""

    features_error = ""

    # --------------------------------------------------------
    # NPZ VALIDATION
    # --------------------------------------------------------

    if graph_file is not None:

        try:

            graph_signature = get_file_signature(
                graph_file
            )

            if graph_signature is None:

                raise FileNotFoundError(
                    f"NPZ file not found: {graph_file}"
                )

            if graph_signature[0] <= 0:

                raise ValueError(
                    "NPZ file is empty."
                )

            # Проверка актуальности.
            #
            # require_fresh=False:
            # допускается использование ранее
            # сохранённого корректного файла.
            #
            # require_fresh=True:
            # требуется изменение сигнатуры
            # относительно состояния до запуска.

            if previous_signatures is None:

                graph_fresh = not require_fresh

            else:

                old_signature = previous_signatures.get(
                    str(graph_file)
                )

                graph_fresh = (
                    graph_signature != old_signature
                )

            # Проверяем возможность чтения архива
            # и каждого массива без использования pickle.

            with np.load(
                graph_file,
                allow_pickle=False
            ) as graph_data:

                graph_keys = list(
                    graph_data.files
                )

                if not graph_keys:

                    raise ValueError(
                        "NPZ archive contains no arrays."
                    )

                for key in graph_keys:

                    array = graph_data[key]

                    if not isinstance(
                        array,
                        np.ndarray
                    ):

                        raise TypeError(
                            f"Invalid NPZ array: {key}"
                        )

            graph_valid = True

        except Exception as exc:

            graph_error = (
                f"{type(exc).__name__}: {exc}"
            )

    else:

        graph_error = (
            "Expected exactly one NPZ file, "
            f"found {len(graph_candidates)}"
        )

    # --------------------------------------------------------
    # JSON VALIDATION
    # --------------------------------------------------------

    if features_json is not None:

        try:

            features_signature = get_file_signature(
                features_json
            )

            if features_signature is None:

                raise FileNotFoundError(
                    f"JSON file not found: {features_json}"
                )

            if features_signature[0] <= 0:

                raise ValueError(
                    "JSON file is empty."
                )

            # Проверка актуальности.

            if previous_signatures is None:

                features_fresh = not require_fresh

            else:

                old_signature = previous_signatures.get(
                    str(features_json)
                )

                features_fresh = (
                    features_signature != old_signature
                )

            # Проверяем корректность JSON.

            with open(
                features_json,
                "r",
                encoding="utf-8"
            ) as file:

                features_data = json.load(file)

            if not isinstance(
                features_data,
                dict
            ):

                raise ValueError(
                    "Features JSON must contain a dictionary."
                )

            features_valid = True

        except Exception as exc:

            features_error = (
                f"{type(exc).__name__}: {exc}"
            )

    else:

        features_error = (
            "Expected exactly one JSON file, "
            f"found {len(feature_candidates)}"
        )

    # --------------------------------------------------------
    # FINAL VALIDATION
    # --------------------------------------------------------

    unique_files = (
        len(graph_candidates) == 1
        and len(feature_candidates) == 1
    )

    # Оба файла должны быть корректными.
    # При require_fresh=True оба должны обновиться.
    # При require_fresh=False старые корректные
    # файлы допускаются без обновления.

    freshness_ok = (

        not require_fresh

        or (

            graph_fresh

            and features_fresh
        )
    )

    valid = (

        unique_files

        and graph_valid

        and features_valid

        and freshness_ok
    )

    messages = []

    if len(graph_candidates) != 1:

        messages.append(
            f"Expected one NPZ, found {len(graph_candidates)}"
        )

    if len(feature_candidates) != 1:

        messages.append(
            f"Expected one JSON, found {len(feature_candidates)}"
        )

    if not graph_valid:

        messages.append(
            f"Invalid NPZ: {graph_error}"
        )

    if not features_valid:

        messages.append(
            f"Invalid JSON: {features_error}"
        )

    if require_fresh and graph_valid and not graph_fresh:

        messages.append(
            "NPZ was not updated during current run"
        )

    if require_fresh and features_valid and not features_fresh:

        messages.append(
            "JSON was not updated during current run"
        )

    return {

        "valid": bool(valid),

        "output_dir": str(pdb_output_dir),

        "graph_file": (
            str(graph_file)
            if graph_file is not None
            else None
        ),

        "features_json": (
            str(features_json)
            if features_json is not None
            else None
        ),

        "graph_valid": bool(graph_valid),

        "features_valid": bool(features_valid),

        "graph_fresh": bool(graph_fresh),

        "features_fresh": bool(features_fresh),

        "graph_candidates_count": len(graph_candidates),

        "features_candidates_count": len(feature_candidates),

        "message": (

            "Required outputs found, readable and updated"

            if valid and require_fresh

            else (

                "Existing required outputs found and readable"

                if valid

                else "; ".join(messages)
            )
        )
    }


# ------------------------------------------------------------
# 8. ATOMIC STATE SAVE
# ------------------------------------------------------------

def save_batch_state(records):

    df = pd.DataFrame(records)

    temporary_path = STATE_CSV.with_suffix(
        ".tmp.csv"
    )

    df.to_csv(
        temporary_path,
        index=False,
        encoding="utf-8-sig"
    )

    # Атомарная замена основного файла состояния.

    temporary_path.replace(
        STATE_CSV
    )


# ------------------------------------------------------------
# 9. RUN ONE PDB
# ------------------------------------------------------------

def run_one_pdb(pdb_id):

    started_at = datetime.now()

    start_time = time.perf_counter()

    log_path = (
        BATCH_LOG_DIR / f"{pdb_id}.log.txt"
    )

    executed_notebook_path = (
        BATCH_NOTEBOOK_DIR / f"{pdb_id}.ipynb"
    )

    status = "FAILED"

    error_message = ""

    notebook_output = ""

    output_validation = {}

    skipped_install_cells = 0

    notebook = None

    notebook_kernel = None

    previous_signatures = {}

    notebook_saved = False

    log_saved = False

    try:

        # ----------------------------------------------------
        # Сохранение состояния файлов ДО запуска.
        # ----------------------------------------------------

        pdb_output_dir = RESULTS_ROOT / pdb_id

        if pdb_output_dir.exists():

            for path in pdb_output_dir.rglob("*"):

                if path.is_file():

                    previous_signatures[str(path)] = (
                        get_file_signature(path)
                    )

        print(
            f"\nPreparing notebook for {pdb_id}..."
        )

        # ----------------------------------------------------
        # Подготовка ноутбука.
        # ----------------------------------------------------

        (
            notebook,
            skipped_install_cells,
            notebook_kernel
        ) = prepare_notebook_for_pdb(
            pdb_id
        )

        print(
            f"[{pdb_id}] Kernel: {notebook_kernel}"
        )

        print(
            f"[{pdb_id}] Skipped installation cells: "
            f"{skipped_install_cells}"
        )

        # ----------------------------------------------------
        # Создание независимого Jupyter Kernel.
        # ----------------------------------------------------

        client = NotebookClient(

            notebook,

            timeout=CELL_TIMEOUT,

            kernel_name=notebook_kernel,

            allow_errors=False,

            resources={

                "metadata": {

                    "path": str(PROJECT_ROOT)
                }
            }
        )

        print(
            f"Starting independent kernel: {pdb_id}"
        )

        # ----------------------------------------------------
        # Выполнение исходного ноутбука.
        # ----------------------------------------------------

        client.execute()

        # ----------------------------------------------------
        # Извлечение вывода.
        # ----------------------------------------------------

        notebook_output = (
            extract_notebook_output(notebook)
        )

        # ----------------------------------------------------
        # Проверка обязательных результатов.
        # Оба файла должны быть обновлены после запуска.
        # ----------------------------------------------------

        output_validation = validate_pdb_outputs(

            pdb_id,

            previous_signatures=previous_signatures,

            require_fresh=True
        )

        if output_validation["valid"]:

            status = "SUCCESS"

            error_message = ""

        else:

            status = "FAILED_OUTPUT_VALIDATION"

            error_message = (
                output_validation["message"]
            )

    except Exception:

        status = "FAILED"

        error_message = traceback.format_exc()

        if notebook is not None:

            try:

                notebook_output = (
                    extract_notebook_output(notebook)
                )

            except Exception as output_exc:

                notebook_output = (
                    "Не удалось извлечь вывод ноутбука: "
                    f"{type(output_exc).__name__}: "
                    f"{output_exc}"
                )

    finally:

        finished_at = datetime.now()

        duration = (
            time.perf_counter() - start_time
        )

        # ----------------------------------------------------
        # Сохранение выполненного ноутбука.
        # ----------------------------------------------------

        if (

            SAVE_EXECUTED_NOTEBOOK

            and notebook is not None
        ):

            try:

                BATCH_NOTEBOOK_DIR.mkdir(
                    parents=True,
                    exist_ok=True
                )

                nbformat.write(
                    notebook,
                    executed_notebook_path
                )

                notebook_saved = True

                print(
                    f"[{pdb_id}] Executed notebook saved."
                )

            except Exception as exc:

                save_error = (

                    "Не удалось сохранить выполненный ноутбук: "

                    f"{type(exc).__name__}: {exc}"
                )

                error_message = (

                    f"{error_message}\n{save_error}"

                    if error_message

                    else save_error
                )

                print(
                    f"[{pdb_id}] WARNING: {save_error}"
                )

        # ----------------------------------------------------
        # Формирование журнала выполнения.
        # ----------------------------------------------------

        log_text = f"""
============================================================
BATCH PROCESSING LOG
============================================================

PDB ID: {pdb_id}

Started: {started_at.isoformat()}
Finished: {finished_at.isoformat()}

Duration: {duration:.2f} seconds

Status: {status}

Kernel: {notebook_kernel}

Skipped installation cells:
{skipped_install_cells}

Executed notebook saved:
{notebook_saved}

Output validation:
{json.dumps(output_validation, ensure_ascii=False, indent=4)}

============================================================
NOTEBOOK OUTPUT
============================================================

{notebook_output}

============================================================
ERROR
============================================================

{error_message}
"""

        # ----------------------------------------------------
        # Устойчивое сохранение журнала.
        # ----------------------------------------------------

        try:

            BATCH_LOG_DIR.mkdir(
                parents=True,
                exist_ok=True
            )

            log_path.write_text(
                log_text,
                encoding="utf-8"
            )

            log_saved = True

        except Exception as exc:

            log_error = (

                "Не удалось сохранить лог: "

                f"{type(exc).__name__}: {exc}"
            )

            error_message = (

                f"{error_message}\n{log_error}"

                if error_message

                else log_error
            )

            print(
                f"[{pdb_id}] WARNING: {log_error}"
            )

    # --------------------------------------------------------
    # Результат обработки одной структуры.
    # --------------------------------------------------------

    return {

        "pdb_id": pdb_id,

        "status": status,

        "started_at": started_at.isoformat(),

        "finished_at": finished_at.isoformat(),

        "duration_seconds": round(duration, 2),

        "output_dir": str(
            RESULTS_ROOT / pdb_id
        ),

        "graph_file": output_validation.get(
            "graph_file"
        ),

        "features_json": output_validation.get(
            "features_json"
        ),

        "error": error_message,

        "log_file": (
            str(log_path)
            if log_saved
            else None
        ),

        "notebook_file": (
            str(executed_notebook_path)
            if notebook_saved
            else None
        ),

        "log_saved": log_saved,

        "notebook_saved": notebook_saved
    }


print("Execution functions initialized.")


Execution functions initialized.


In [5]:

# ============================================================
# CELL 5. BATCH EXECUTION ENGINE
# ============================================================
# Назначение:
# 1. Последовательная обработка PDB.
# 2. Пропуск успешно обработанных структур.
# 3. Повтор неудачных запусков по настройке.
# 4. Регистрация времени и ошибок.
# 5. Сохранение состояния после каждого PDB.
# 6. Исключение дублирующихся записей в истории.
# 7. Восстановление состояния после прерывания.
# 8. Итоговая статистика выполнения.
# ============================================================


# ------------------------------------------------------------
# 1. RESTORE PREVIOUS RECORDS
# ------------------------------------------------------------

batch_records = []

for _, row in previous_state.iterrows():

    batch_records.append(
        row.to_dict()
    )


# ------------------------------------------------------------
# 2. CURRENT STATUS MAP
# ------------------------------------------------------------

def get_latest_status_map(records):

    result = {}

    for record in records:

        pdb_id = str(
            record.get("pdb_id", "")
        ).strip().upper()

        if not pdb_id:
            continue

        record = dict(record)

        record["pdb_id"] = pdb_id

        # Последняя запись для PDB заменяет предыдущую.
        result[pdb_id] = record

    return result


status_map = get_latest_status_map(
    batch_records
)

# Убираем дубли из истории.
batch_records = list(
    status_map.values()
)


# ------------------------------------------------------------
# 2.1. NORMALIZE AND DEDUPLICATE PDB IDS
# ------------------------------------------------------------

PDB_IDS_UNIQUE = list(
    dict.fromkeys(
        str(pdb_id).strip().upper()
        for pdb_id in PDB_IDS
        if str(pdb_id).strip()
    )
)

print(
    "Unique PDB count:",
    len(PDB_IDS_UNIQUE)
)

if len(PDB_IDS_UNIQUE) != len(PDB_IDS):

    print(
        "WARNING: Duplicate or empty PDB IDs were removed."
    )


# ------------------------------------------------------------
# 3. BATCH EXECUTION
# ------------------------------------------------------------

batch_start_time = time.perf_counter()

batch_started_at = datetime.now()

execution_results = []

batch_interrupted = False

current_pdb_id = None

print("=" * 75)
print("BATCH EXECUTION STARTED")
print("=" * 75)

print("PDB count:", len(PDB_IDS_UNIQUE))
print("Mode:", BATCH_MODE)
print("Kernel:", KERNEL_NAME)
print("Run directory:", BATCH_RUN_DIR)
print("Skip completed:", SKIP_COMPLETED)
print("Retry failed:", RETRY_FAILED)

print("=" * 75)


try:

    for pdb_id in tqdm(
        PDB_IDS_UNIQUE,
        desc="Processing PDB structures",
        unit="PDB"
    ):

        pdb_id = str(
            pdb_id
        ).strip().upper()

        current_pdb_id = pdb_id

        previous_record = status_map.get(
            pdb_id
        )

        previous_status = (

            str(
                previous_record.get("status", "")
            ).strip().upper()

            if previous_record is not None

            else ""
        )

        # ----------------------------------------------------
        # CHECK PREVIOUS SUCCESS
        # ----------------------------------------------------

        if (

            SKIP_COMPLETED

            and previous_record is not None

            and previous_status == "SUCCESS"
        ):

            # Для уже обработанных структур не требуется
            # обновление файлов в текущем запуске.
            # Проверяется только наличие и корректность файлов.

            validation = validate_pdb_outputs(
                pdb_id,
                require_fresh=False
            )

            if validation["valid"]:

                print(
                    f"\n[{pdb_id}] SKIPPED: already completed."
                )

                execution_results.append({

                    **previous_record,

                    "status": "SKIPPED_COMPLETED"
                })

                # Не изменяем status_map:
                # в постоянной истории остаётся SUCCESS.

                continue

            else:

                print(
                    f"\n[{pdb_id}] Previous SUCCESS has invalid outputs."
                )

                print(
                    "The structure will be processed again."
                )

                print(
                    "Validation:",
                    validation["message"]
                )

        # ----------------------------------------------------
        # CHECK PREVIOUS FAILURE
        # ----------------------------------------------------

        if (

            previous_record is not None

            and previous_status.startswith("FAILED")

            and not RETRY_FAILED
        ):

            print(
                f"\n[{pdb_id}] SKIPPED: previous failure."
            )

            execution_results.append({

                **previous_record,

                "status": "SKIPPED_PREVIOUS_FAILURE"
            })

            # Сохраняем исходный FAILED в истории.
            continue

        # ----------------------------------------------------
        # RUN CURRENT PDB
        # ----------------------------------------------------

        print("\n" + "=" * 75)

        print(
            f"PROCESSING: {pdb_id}"
        )

        print("=" * 75)

        # ----------------------------------------------------
        # Защита основного цикла от ошибки одной структуры.
        # ----------------------------------------------------

        try:

            result = run_one_pdb(
                pdb_id
            )

        except Exception:

            # Дополнительная защита на случай,
            # если run_one_pdb не смог вернуть результат.

            error_message = traceback.format_exc()

            result = {

                "pdb_id": pdb_id,

                "status": "FAILED",

                "started_at": datetime.now().isoformat(),

                "finished_at": datetime.now().isoformat(),

                "duration_seconds": 0.0,

                "output_dir": str(
                    RESULTS_ROOT / pdb_id
                ),

                "graph_file": None,

                "features_json": None,

                "error": error_message,

                "log_file": None,

                "notebook_file": None,

                "log_saved": False,

                "notebook_saved": False
            }

            print(
                f"[{pdb_id}] ERROR: run_one_pdb failed."
            )

            print(error_message)

        # ----------------------------------------------------
        # REGISTER RESULT
        # ----------------------------------------------------

        execution_results.append(
            result
        )

        # Обновляем актуальную запись.
        status_map[pdb_id] = result

        # Формируем историю без дублирующихся PDB.
        batch_records = list(
            status_map.values()
        )

        # ----------------------------------------------------
        # SAVE STATE AFTER EACH PDB
        # ----------------------------------------------------

        try:

            save_batch_state(
                batch_records
            )

            print(
                f"[{pdb_id}] Batch state saved."
            )

        except Exception:

            state_error = traceback.format_exc()

            print(
                f"[{pdb_id}] CRITICAL: state save failed."
            )

            print(state_error)

            # Не продолжаем большой batch,
            # если невозможно сохранить состояние.
            raise RuntimeError(
                f"Не удалось сохранить состояние после {pdb_id}."
            )

        # ----------------------------------------------------
        # PRINT CURRENT RESULT
        # ----------------------------------------------------

        print(
            f"\n[{pdb_id}] STATUS: {result['status']}"
        )

        print(
            f"Duration: {result['duration_seconds']} sec"
        )

        if result["status"] != "SUCCESS":

            print(
                f"Log: {result.get('log_file')}"
            )

            if result.get("error"):

                print(
                    "Error:",
                    result["error"]
                )


except KeyboardInterrupt:

    batch_interrupted = True

    print("\n" + "=" * 75)

    print("BATCH INTERRUPTED BY USER")

    print("=" * 75)

    # Уже завершённые структуры остаются в state.
    # Текущая незавершённая структура при следующем
    # запуске будет обработана повторно.


except Exception:

    batch_interrupted = True

    print("\n" + "=" * 75)

    print("BATCH STOPPED DUE TO CRITICAL ERROR")

    print("=" * 75)

    print(
        traceback.format_exc()
    )


finally:

    # --------------------------------------------------------
    # 4. FINAL EXECUTION STATE
    # --------------------------------------------------------

    batch_finished_at = datetime.now()

    batch_duration = (
        time.perf_counter() - batch_start_time
    )

    # Финальная история состояния.
    batch_records = list(
        status_map.values()
    )

    batch_state_df = pd.DataFrame(
        batch_records
    )

    # Финальное атомарное сохранение.
    try:

        save_batch_state(
            batch_records
        )

        print(
            "\nFinal batch state saved."
        )

    except Exception:

        print(
            "\nCRITICAL: final state save failed."
        )

        print(
            traceback.format_exc()
        )

    # --------------------------------------------------------
    # 5. EXECUTION SUMMARY
    # --------------------------------------------------------

    execution_summary = pd.DataFrame(
        execution_results
    )

    if not execution_summary.empty:

        summary_counts = (
            execution_summary["status"]
            .value_counts()
        )

    else:

        summary_counts = pd.Series(
            dtype=int
        )

    print("\n" + "=" * 75)

    if batch_interrupted:

        print(
            "BATCH EXECUTION STOPPED"
        )

    else:

        print(
            "BATCH EXECUTION FINISHED"
        )

    print("=" * 75)

    print(
        "Started:",
        batch_started_at
    )

    print(
        "Finished:",
        batch_finished_at
    )

    print(
        f"Total duration: {batch_duration / 60:.2f} minutes"
    )

    print(
        "Total unique PDB IDs:",
        len(PDB_IDS_UNIQUE)
    )

    print(
        "Entries in current run:",
        len(execution_results)
    )

    print("\nCURRENT RUN STATUS:")

    if not summary_counts.empty:

        print(
            summary_counts.to_string()
        )

    else:

        print(
            "No structures processed."
        )

    print(
        "\nTOTAL STRUCTURES IN STATE:",
        len(batch_state_df)
    )

    print(
        "State file:",
        STATE_CSV
    )

    print("=" * 75)


Unique PDB count: 100
BATCH EXECUTION STARTED
PDB count: 100
Mode: BATCH
Kernel: python3
Run directory: C:\apex\APEX\МИФИ\СНО\medic\Protein_Pocket_Dataset\07_structure_analysis\batch_runs\protein_batch_v1
Skip completed: True
Retry failed: True


Processing PDB structures:   0%|          | 0/100 [00:00<?, ?PDB/s]


[7WTU] SKIPPED: already completed.

[7W37] SKIPPED: already completed.

[7Q0G] SKIPPED: already completed.

[7VYS] SKIPPED: already completed.

[7X8R] SKIPPED: already completed.

[7QB5] SKIPPED: already completed.

[7Z0O] SKIPPED: already completed.

[7XCR] SKIPPED: already completed.

[12TB] SKIPPED: already completed.

[3PUW] SKIPPED: already completed.

[7WTT] SKIPPED: already completed.

[3O2R] SKIPPED: already completed.

[3NV0] SKIPPED: already completed.

[7Z16] SKIPPED: already completed.

[7WSO] SKIPPED: already completed.

[7PR4] SKIPPED: already completed.

[7W38] SKIPPED: already completed.

[3OFG] SKIPPED: already completed.

[7XNX] SKIPPED: already completed.

[3P1A] SKIPPED: already completed.

[7W1Z] SKIPPED: already completed.

[7OYK] SKIPPED: already completed.

[7WIG] SKIPPED: already completed.

[7PS6] SKIPPED: already completed.

[7W2U] SKIPPED: already completed.

[3NZJ] SKIPPED: already completed.

[7XCT] SKIPPED: already completed.

[7YMS] SKIPPED: already com

In [8]:

# ============================================================
# CELL 6. BATCH QUALITY CONTROL
# ============================================================
# Назначение:
# 1. Проверка итоговых файлов каждого PDB.
# 2. Проверка наличия NPZ графа.
# 3. Проверка JSON с признаками белка.
# 4. Формирование итоговой таблицы качества.
# 5. Сохранение результатов QC.
#
# ВАЖНО:
# CELL 4 уже проверяет обновление файлов
# непосредственно после обработки каждой структуры.
# Здесь повторно проверяем наличие и корректность
# сохранённых результатов без требования обновления.
# ============================================================


qc_records = []

for pdb_id in PDB_IDS:

    output_check = validate_pdb_outputs(
        pdb_id,
        require_fresh=False
    )

    current_status = status_map.get(
        pdb_id,
        {}
    )

    qc_records.append({

        "pdb_id": pdb_id,

        "processing_status": current_status.get(
            "status",
            "NOT_PROCESSED"
        ),

        "graph_file_exists": (
            output_check["graph_file"] is not None
        ),

        "features_json_exists": (
            output_check["features_json"] is not None
        ),

        "outputs_valid": output_check["valid"],

        "output_dir": output_check["output_dir"],

        "graph_file": output_check["graph_file"],

        "features_json": output_check["features_json"],

        "qc_message": output_check["message"]

    })


df_batch_qc = pd.DataFrame(
    qc_records
)


# ------------------------------------------------------------
# SAVE QC
# ------------------------------------------------------------

QC_CSV = (
    BATCH_RUN_DIR / "batch_quality_control.csv"
)

df_batch_qc.to_csv(
    QC_CSV,
    index=False,
    encoding="utf-8-sig"
)


# ------------------------------------------------------------
# STATUS COUNTS
# ------------------------------------------------------------

status_counts = (
    df_batch_qc["processing_status"]
    .value_counts(dropna=False)
    .to_dict()
)

valid_count = int(
    df_batch_qc["outputs_valid"].sum()
)

invalid_count = (
    len(df_batch_qc) - valid_count
)


print("=" * 75)
print("BATCH QUALITY CONTROL")
print("=" * 75)

print("Total structures:", len(df_batch_qc))
print("Valid output sets:", valid_count)
print("Invalid output sets:", invalid_count)

print("-" * 75)

print("PROCESSING STATUS")

for status, count in status_counts.items():

    print(
        f"{str(status):35s}: {count}"
    )

print("-" * 75)

display(
    df_batch_qc
)

print("-" * 75)

print("QC file:", QC_CSV)

print("=" * 75)


BATCH QUALITY CONTROL
Total structures: 100
Valid output sets: 100
Invalid output sets: 0
---------------------------------------------------------------------------
PROCESSING STATUS
SUCCESS                            : 100
---------------------------------------------------------------------------


,pdb_id,processing_status,graph_file_exists,features_json_exists,outputs_valid,output_dir,graph_file,features_json,qc_message
0,7WTU,SUCCESS,True,True,True,C:\apex\APEX\МИФИ\СНО\medic\Protein_Pocket_Dat...,C:\apex\APEX\МИФИ\СНО\medic\Protein_Pocket_Dat...,C:\apex\APEX\МИФИ\СНО\medic\Protein_Pocket_Dat...,Existing required outputs found and readable
1,7W37,SUCCESS,True,True,True,C:\apex\APEX\МИФИ\СНО\medic\Protein_Pocket_Dat...,C:\apex\APEX\МИФИ\СНО\medic\Protein_Pocket_Dat...,C:\apex\APEX\МИФИ\СНО\medic\Protein_Pocket_Dat...,Existing required outputs found and readable
2,7Q0G,SUCCESS,True,True,True,C:\apex\APEX\МИФИ\СНО\medic\Protein_Pocket_Dat...,C:\apex\APEX\МИФИ\СНО\medic\Protein_Pocket_Dat...,C:\apex\APEX\МИФИ\СНО\medic\Protein_Pocket_Dat...,Existing required outputs found and readable
3,7VYS,SUCCESS,True,True,True,C:\apex\APEX\МИФИ\СНО\medic\Protein_Pocket_Dat...,C:\apex\APEX\МИФИ\СНО\medic\Protein_Pocket_Dat...,C:\apex\APEX\МИФИ\СНО\medic\Protein_Pocket_Dat...,Existing required outputs found and readable
4,7X8R,SUCCESS,True,True,True,C:\apex\APEX\МИФИ\СНО\medic\Protein_Pocket_Dat...,C:\apex\APEX\МИФИ\СНО\medic\Protein_Pocket_Dat...,C:\apex\APEX\МИФИ\СНО\medic\Protein_Pocket_Dat...,Existing required outputs found and readable
...,...,...,...,...,...,...,...,...,...
95,7WU3,SUCCESS,True,True,True,C:\apex\APEX\МИФИ\СНО\medic\Protein_Pocket_Dat...,C:\apex\APEX\МИФИ\СНО\medic\Protein_Pocket_Dat...,C:\apex\APEX\МИФИ\СНО\medic\Protein_Pocket_Dat...,Existing required outputs found and readable
96,3PAM,SUCCESS,True,True,True,C:\apex\APEX\МИФИ\СНО\medic\Protein_Pocket_Dat...,C:\apex\APEX\МИФИ\СНО\medic\Protein_Pocket_Dat...,C:\apex\APEX\МИФИ\СНО\medic\Protein_Pocket_Dat...,Existing required outputs found and readable
97,7XD1,SUCCESS,True,True,True,C:\apex\APEX\МИФИ\СНО\medic\Protein_Pocket_Dat...,C:\apex\APEX\МИФИ\СНО\medic\Protein_Pocket_Dat...,C:\apex\APEX\МИФИ\СНО\medic\Protein_Pocket_Dat...,Existing required outputs found and readable
98,7P0J,SUCCESS,True,True,True,C:\apex\APEX\МИФИ\СНО\medic\Protein_Pocket_Dat...,C:\apex\APEX\МИФИ\СНО\medic\Protein_Pocket_Dat...,C:\apex\APEX\МИФИ\СНО\medic\Protein_Pocket_Dat...,Existing required outputs found and readable


---------------------------------------------------------------------------
QC file: C:\apex\APEX\МИФИ\СНО\medic\Protein_Pocket_Dataset\07_structure_analysis\batch_runs\protein_batch_v1\batch_quality_control.csv


In [7]:

# ============================================================
# CELL 7. FINAL BATCH REPORT
# ============================================================
# Назначение:
# 1. Формирование итогового реестра пакетной обработки.
# 2. Подсчёт успешно обработанных структур.
# 3. Учёт ранее завершённых и пропущенных структур.
# 4. Подсчёт ошибок.
# 5. Оценка полноты выходных файлов.
# 6. Сохранение CSV и JSON-манифеста.
# ============================================================


# ============================================================
# 1. PREPARE FINAL QC DATA
# ============================================================

df_batch_final = df_batch_qc.copy()

total_structures = len(df_batch_final)

if total_structures == 0:
    raise ValueError(
        "Итоговая таблица QC пуста. "
        "Невозможно сформировать отчёт."
    )


# ============================================================
# 2. NORMALIZE STATUS VALUES
# ============================================================

df_batch_final["processing_status"] = (
    df_batch_final["processing_status"]
    .fillna("UNKNOWN")
    .astype(str)
    .str.strip()
    .str.upper()
)

df_batch_final["outputs_valid"] = (
    df_batch_final["outputs_valid"]
    .fillna(False)
    .astype(bool)
)


# ============================================================
# 3. CALCULATE PROCESSING STATISTICS
# ============================================================

# SUCCESS — успешно обработаны в текущем или
# предыдущем запуске.

# SKIPPED_COMPLETED — ранее обработаны и пропущены
# при текущем запуске.

# SKIPPED_PREVIOUS_FAILURE — не запускались повторно,
# поскольку RETRY_FAILED = False.

completed_statuses = {
    "SUCCESS",
    "SKIPPED_COMPLETED"
}

successful_structures = int(
    df_batch_final["processing_status"]
    .isin(completed_statuses)
    .sum()
)

skipped_completed = int(
    (
        df_batch_final["processing_status"]
        == "SKIPPED_COMPLETED"
    ).sum()
)

skipped_previous_failure = int(
    (
        df_batch_final["processing_status"]
        == "SKIPPED_PREVIOUS_FAILURE"
    ).sum()
)

failed_structures = int(
    df_batch_final["processing_status"]
    .str.startswith("FAILED")
    .sum()
)

not_processed = int(
    (
        df_batch_final["processing_status"]
        == "NOT_PROCESSED"
    ).sum()
)

unknown_status = int(
    (
        df_batch_final["processing_status"]
        == "UNKNOWN"
    ).sum()
)


# ============================================================
# 4. OUTPUT VALIDATION STATISTICS
# ============================================================

valid_outputs = int(
    df_batch_final["outputs_valid"].sum()
)

invalid_outputs = (
    total_structures - valid_outputs
)

success_rate = (
    successful_structures / total_structures * 100
)

output_validity_rate = (
    valid_outputs / total_structures * 100
)


# ============================================================
# 5. DURATION STATISTICS
# ============================================================

if "duration_seconds" in batch_state_df.columns:

    duration_values = pd.to_numeric(
        batch_state_df["duration_seconds"],
        errors="coerce"
    )

    duration_values = duration_values.dropna()

else:

    duration_values = pd.Series(
        dtype=float
    )


if not duration_values.empty:

    mean_duration = float(
        duration_values.mean()
    )

    median_duration = float(
        duration_values.median()
    )

    max_duration = float(
        duration_values.max()
    )

    min_duration = float(
        duration_values.min()
    )

else:

    mean_duration = None
    median_duration = None
    max_duration = None
    min_duration = None


# ============================================================
# 6. BUILD STATUS DISTRIBUTION
# ============================================================

status_distribution = (
    df_batch_final["processing_status"]
    .value_counts(dropna=False)
    .to_dict()
)

status_distribution = {
    str(key): int(value)
    for key, value in status_distribution.items()
}


# ============================================================
# 7. BUILD FINAL STRUCTURE REGISTRY
# ============================================================

# Объединяем сведения QC с актуальным состоянием
# пакетной обработки.

df_batch_registry = df_batch_final.copy()

df_batch_registry["batch_run_name"] = (
    BATCH_RUN_NAME
)

df_batch_registry["batch_mode"] = (
    BATCH_MODE
)

df_batch_registry["report_generated_at"] = (
    datetime.now().isoformat()
)


# ============================================================
# 8. SAVE FINAL CSV
# ============================================================

df_batch_registry.to_csv(
    SUMMARY_CSV,
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 9. BUILD JSON MANIFEST
# ============================================================

manifest = {

    "batch_run_name": BATCH_RUN_NAME,

    "batch_mode": BATCH_MODE,

    "started_at": batch_started_at.isoformat(),

    "finished_at": batch_finished_at.isoformat(),

    "report_generated_at": datetime.now().isoformat(),

    "total_duration_seconds": round(
        float(batch_duration),
        2
    ),

    "processing_statistics": {

        "total_structures": total_structures,

        "successful_structures": successful_structures,

        "skipped_completed": skipped_completed,

        "skipped_previous_failure": skipped_previous_failure,

        "failed_structures": failed_structures,

        "not_processed": not_processed,

        "unknown_status": unknown_status,

        "success_rate_percent": round(
            success_rate,
            2
        )

    },

    "output_validation": {

        "valid_output_sets": valid_outputs,

        "invalid_output_sets": invalid_outputs,

        "output_validity_rate_percent": round(
            output_validity_rate,
            2
        )

    },

    "duration_statistics": {

        "mean_seconds": mean_duration,

        "median_seconds": median_duration,

        "min_seconds": min_duration,

        "max_seconds": max_duration

    },

    "status_distribution": status_distribution,

    "source_notebook": str(
        ANALYZER_NOTEBOOK_PATH
    ),

    "selection_csv": str(
        SELECTION_CSV
    ),

    "results_root": str(
        RESULTS_ROOT
    ),

    "batch_run_directory": str(
        BATCH_RUN_DIR
    ),

    "processed_pdb_ids": [
        str(x)
        for x in PDB_IDS
    ],

    "component_selection_registry": (
        COMPONENT_SELECTION_BY_PDB
    ),

    "output_files": {

        "state_csv": str(STATE_CSV),

        "quality_control_csv": str(QC_CSV),

        "summary_csv": str(SUMMARY_CSV),

        "manifest_json": str(MANIFEST_JSON)

    },

    "limitations": [

        "This batch run executes Protein Structure Analyzer only.",

        "Pocket Detection and Protein-Pocket Integration "
        "are not yet connected to this controller.",

        "Component selection requires a per-PDB registry.",

        "Output-file validation confirms file presence and "
        "non-zero file size, not scientific correctness.",

        "A skipped structure is considered completed only "
        "when its required output files pass validation."

    ]

}


# ============================================================
# 10. JSON SAFE CONVERSION
# ============================================================

def batch_json_safe(value):

    if isinstance(value, dict):

        return {
            str(key): batch_json_safe(item)
            for key, item in value.items()
        }

    if isinstance(value, (list, tuple)):

        return [
            batch_json_safe(item)
            for item in value
        ]

    if isinstance(value, Path):

        return str(value)

    if isinstance(value, (np.integer,)):

        return int(value)

    if isinstance(value, (np.floating,)):

        return (
            float(value)
            if np.isfinite(value)
            else None
        )

    if isinstance(value, (np.bool_,)):

        return bool(value)

    if isinstance(value, float):

        return (
            value
            if np.isfinite(value)
            else None
        )

    if value is pd.NA or value is pd.NaT:

        return None

    if isinstance(value, (datetime,)):

        return value.isoformat()

    return value


manifest = batch_json_safe(
    manifest
)


# ============================================================
# 11. SAVE JSON MANIFEST
# ============================================================

temporary_manifest = (
    MANIFEST_JSON.with_suffix(".tmp.json")
)

with open(
    temporary_manifest,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        manifest,
        file,
        ensure_ascii=False,
        indent=4,
        allow_nan=False
    )

temporary_manifest.replace(
    MANIFEST_JSON
)


# ============================================================
# 12. FINAL REPORT
# ============================================================

print("\n" + "=" * 75)
print("FINAL BATCH PROCESSING REPORT")
print("=" * 75)

print(f"Batch run:                 {BATCH_RUN_NAME}")
print(f"Processing mode:           {BATCH_MODE}")

print("-" * 75)

print(f"Total structures:          {total_structures}")
print(f"Successful structures:     {successful_structures}")
print(f"Skipped completed:         {skipped_completed}")
print(f"Skipped previous failure:  {skipped_previous_failure}")
print(f"Failed structures:         {failed_structures}")
print(f"Not processed:             {not_processed}")
print(f"Unknown status:            {unknown_status}")

print("-" * 75)

print(f"Valid output sets:          {valid_outputs}")
print(f"Invalid output sets:        {invalid_outputs}")

print(f"Success rate:               {success_rate:.2f}%")
print(f"Output validity rate:       {output_validity_rate:.2f}%")

print("-" * 75)

if mean_duration is not None:

    print(
        f"Mean PDB duration:          {mean_duration:.2f} sec"
    )

    print(
        f"Median PDB duration:        {median_duration:.2f} sec"
    )

    print(
        f"Minimum PDB duration:       {min_duration:.2f} sec"
    )

    print(
        f"Maximum PDB duration:       {max_duration:.2f} sec"
    )

else:

    print("Duration statistics:       unavailable")


print("-" * 75)

print("OUTPUT FILES")

print("State CSV:       ", STATE_CSV)
print("QC CSV:          ", QC_CSV)
print("Summary CSV:     ", SUMMARY_CSV)
print("Manifest JSON:   ", MANIFEST_JSON)

print("=" * 75)


# ============================================================
# 13. FINAL STATUS
# ============================================================

if failed_structures > 0:

    print(
        "WARNING: обнаружены структуры с ошибками."
    )

elif invalid_outputs > 0:

    print(
        "WARNING: часть структур не прошла проверку выходных файлов."
    )

elif not_processed > 0:

    print(
        "WARNING: имеются необработанные структуры."
    )

else:

    print(
        "BATCH PROCESSING COMPLETED."
    )


print("=" * 75)


# ============================================================
# 14. DISPLAY FINAL TABLE
# ============================================================

display(
    df_batch_registry
)



FINAL BATCH PROCESSING REPORT
Batch run:                 protein_batch_v1
Processing mode:           BATCH
---------------------------------------------------------------------------
Total structures:          100
Successful structures:     100
Skipped completed:         0
Skipped previous failure:  0
Failed structures:         0
Not processed:             0
Unknown status:            0
---------------------------------------------------------------------------
Valid output sets:          0
Invalid output sets:        100
Success rate:               100.00%
Output validity rate:       0.00%
---------------------------------------------------------------------------
Mean PDB duration:          43.07 sec
Median PDB duration:        17.96 sec
Minimum PDB duration:       10.42 sec
Maximum PDB duration:       1422.66 sec
---------------------------------------------------------------------------
OUTPUT FILES
State CSV:        C:\apex\APEX\МИФИ\СНО\medic\Protein_Pocket_Dataset\07_structure_

,pdb_id,processing_status,graph_file_exists,features_json_exists,outputs_valid,output_dir,graph_file,features_json,qc_message,batch_run_name,batch_mode,report_generated_at
0,7WTU,SUCCESS,True,True,False,C:\apex\APEX\МИФИ\СНО\medic\Protein_Pocket_Dat...,C:\apex\APEX\МИФИ\СНО\medic\Protein_Pocket_Dat...,C:\apex\APEX\МИФИ\СНО\medic\Protein_Pocket_Dat...,NPZ was not updated during current run; JSON w...,protein_batch_v1,BATCH,2026-11-07T14:46:28.159102
1,7W37,SUCCESS,True,True,False,C:\apex\APEX\МИФИ\СНО\medic\Protein_Pocket_Dat...,C:\apex\APEX\МИФИ\СНО\medic\Protein_Pocket_Dat...,C:\apex\APEX\МИФИ\СНО\medic\Protein_Pocket_Dat...,NPZ was not updated during current run; JSON w...,protein_batch_v1,BATCH,2026-11-07T14:46:28.159102
2,7Q0G,SUCCESS,True,True,False,C:\apex\APEX\МИФИ\СНО\medic\Protein_Pocket_Dat...,C:\apex\APEX\МИФИ\СНО\medic\Protein_Pocket_Dat...,C:\apex\APEX\МИФИ\СНО\medic\Protein_Pocket_Dat...,NPZ was not updated during current run; JSON w...,protein_batch_v1,BATCH,2026-11-07T14:46:28.159102
3,7VYS,SUCCESS,True,True,False,C:\apex\APEX\МИФИ\СНО\medic\Protein_Pocket_Dat...,C:\apex\APEX\МИФИ\СНО\medic\Protein_Pocket_Dat...,C:\apex\APEX\МИФИ\СНО\medic\Protein_Pocket_Dat...,NPZ was not updated during current run; JSON w...,protein_batch_v1,BATCH,2026-11-07T14:46:28.159102
4,7X8R,SUCCESS,True,True,False,C:\apex\APEX\МИФИ\СНО\medic\Protein_Pocket_Dat...,C:\apex\APEX\МИФИ\СНО\medic\Protein_Pocket_Dat...,C:\apex\APEX\МИФИ\СНО\medic\Protein_Pocket_Dat...,NPZ was not updated during current run; JSON w...,protein_batch_v1,BATCH,2026-11-07T14:46:28.159102
...,...,...,...,...,...,...,...,...,...,...,...,...
95,7WU3,SUCCESS,True,True,False,C:\apex\APEX\МИФИ\СНО\medic\Protein_Pocket_Dat...,C:\apex\APEX\МИФИ\СНО\medic\Protein_Pocket_Dat...,C:\apex\APEX\МИФИ\СНО\medic\Protein_Pocket_Dat...,NPZ was not updated during current run; JSON w...,protein_batch_v1,BATCH,2026-11-07T14:46:28.159102
96,3PAM,SUCCESS,True,True,False,C:\apex\APEX\МИФИ\СНО\medic\Protein_Pocket_Dat...,C:\apex\APEX\МИФИ\СНО\medic\Protein_Pocket_Dat...,C:\apex\APEX\МИФИ\СНО\medic\Protein_Pocket_Dat...,NPZ was not updated during current run; JSON w...,protein_batch_v1,BATCH,2026-11-07T14:46:28.159102
97,7XD1,SUCCESS,True,True,False,C:\apex\APEX\МИФИ\СНО\medic\Protein_Pocket_Dat...,C:\apex\APEX\МИФИ\СНО\medic\Protein_Pocket_Dat...,C:\apex\APEX\МИФИ\СНО\medic\Protein_Pocket_Dat...,NPZ was not updated during current run; JSON w...,protein_batch_v1,BATCH,2026-11-07T14:46:28.159102
98,7P0J,SUCCESS,True,True,False,C:\apex\APEX\МИФИ\СНО\medic\Protein_Pocket_Dat...,C:\apex\APEX\МИФИ\СНО\medic\Protein_Pocket_Dat...,C:\apex\APEX\МИФИ\СНО\medic\Protein_Pocket_Dat...,NPZ was not updated during current run; JSON w...,protein_batch_v1,BATCH,2026-11-07T14:46:28.159102


In [9]:

# ============================================================
# CELL 8. CONTENT AUDIT OF NPZ AND JSON OUTPUTS
# ============================================================
# Назначение:
# 1. Проверить содержимое NPZ-файлов всех PDB.
# 2. Проверить содержимое JSON-файлов всех PDB.
# 3. Получить информацию о массивах, размерах и типах данных.
# 4. Найти NaN, Inf, пустые массивы и пустые JSON-поля.
# 5. Сравнить структуру JSON между всеми PDB.
# 6. Сохранить диагностические отчёты.
#
# ВАЖНО:
# - Исходные NPZ и JSON не изменяются.
# - Файлы читаются только для анализа.
# - NPZ открываются с allow_pickle=False.
# - CELL 17 и CELL 18 не запускаются.
# ============================================================

import json
import numpy as np
import pandas as pd

from pathlib import Path
from collections import Counter
from datetime import datetime
import traceback


# ============================================================
# 1. CHECK INPUT DATA
# ============================================================

print("=" * 75)
print("CELL 8. CONTENT AUDIT OF NPZ AND JSON OUTPUTS")
print("=" * 75)

if "df_batch_qc" not in globals():
    raise RuntimeError(
        "df_batch_qc не найден. Сначала выполните CELL 6."
    )

required_columns = {
    "pdb_id",
    "graph_file",
    "features_json",
    "outputs_valid"
}

missing_columns = required_columns - set(df_batch_qc.columns)

if missing_columns:
    raise RuntimeError(
        f"В df_batch_qc отсутствуют столбцы: {sorted(missing_columns)}"
    )

audit_df = df_batch_qc.copy()

print(f"Structures to audit: {len(audit_df)}")
print(f"Started: {datetime.now().isoformat(timespec='seconds')}")


# ============================================================
# 2. HELPER FUNCTIONS
# ============================================================

def is_empty_value(value):
    """Определяет пустое JSON-значение."""

    if value is None:
        return True

    if isinstance(value, str):
        return len(value.strip()) == 0

    if isinstance(value, (list, dict)):
        return len(value) == 0

    return False


def flatten_json(data, prefix=""):
    """
    Рекурсивно формирует список JSON-полей.

    Значения полей не сохраняются:
    фиксируются только путь, тип и размер.
    """

    records = []

    if isinstance(data, dict):

        if not data:
            records.append({
                "json_path": prefix or "$",
                "value_type": "dict",
                "is_empty": True,
                "item_count": 0
            })

        for key, value in data.items():

            key_str = str(key)

            current_path = (
                f"{prefix}.{key_str}"
                if prefix else key_str
            )

            if isinstance(value, (dict, list)):
                records.append({
                    "json_path": current_path,
                    "value_type": type(value).__name__,
                    "is_empty": is_empty_value(value),
                    "item_count": len(value)
                })

                records.extend(
                    flatten_json(value, current_path)
                )

            else:
                records.append({
                    "json_path": current_path,
                    "value_type": type(value).__name__,
                    "is_empty": is_empty_value(value),
                    "item_count": None
                })

    elif isinstance(data, list):

        if not data:
            records.append({
                "json_path": prefix or "$",
                "value_type": "list",
                "is_empty": True,
                "item_count": 0
            })

        for index, value in enumerate(data):

            current_path = f"{prefix}[{index}]"

            if isinstance(value, (dict, list)):
                records.extend(
                    flatten_json(value, current_path)
                )
            else:
                records.append({
                    "json_path": current_path,
                    "value_type": type(value).__name__,
                    "is_empty": is_empty_value(value),
                    "item_count": None
                })

    else:
        records.append({
            "json_path": prefix or "$",
            "value_type": type(data).__name__,
            "is_empty": is_empty_value(data),
            "item_count": None
        })

    return records


def inspect_npz(npz_path):
    """Проверяет NPZ и возвращает информацию о каждом массиве."""

    result = {
        "readable": False,
        "array_count": 0,
        "total_elements": 0,
        "empty_arrays": 0,
        "nan_count": 0,
        "inf_count": 0,
        "object_arrays": 0,
        "arrays": [],
        "error": None
    }

    if not npz_path:
        result["error"] = "NPZ path is missing"
        return result

    path = Path(str(npz_path))

    if not path.is_file():
        result["error"] = "NPZ file does not exist"
        return result

    try:
        with np.load(path, allow_pickle=False) as archive:

            result["array_count"] = len(archive.files)

            for array_name in archive.files:

                try:
                    array = archive[array_name]

                    shape = tuple(array.shape)
                    dtype = str(array.dtype)
                    size = int(array.size)

                    is_numeric = np.issubdtype(
                        array.dtype,
                        np.number
                    )

                    nan_count = 0
                    inf_count = 0

                    if is_numeric:

                        if np.issubdtype(
                            array.dtype,
                            np.complexfloating
                        ):
                            nan_count = int(
                                np.isnan(array).sum()
                            )
                            inf_count = int(
                                np.isinf(array).sum()
                            )

                        elif np.issubdtype(
                            array.dtype,
                            np.floating
                        ):
                            nan_count = int(
                                np.isnan(array).sum()
                            )
                            inf_count = int(
                                np.isinf(array).sum()
                            )

                    is_empty = size == 0

                    result["total_elements"] += size
                    result["empty_arrays"] += int(is_empty)
                    result["nan_count"] += nan_count
                    result["inf_count"] += inf_count

                    if array.dtype.hasobject:
                        result["object_arrays"] += 1

                    result["arrays"].append({
                        "array_name": array_name,
                        "shape": str(shape),
                        "dtype": dtype,
                        "elements": size,
                        "is_empty": is_empty,
                        "is_numeric": is_numeric,
                        "nan_count": nan_count,
                        "inf_count": inf_count,
                        "read_error": None
                    })

                except Exception as exc:

                    result["arrays"].append({
                        "array_name": array_name,
                        "shape": None,
                        "dtype": None,
                        "elements": None,
                        "is_empty": None,
                        "is_numeric": None,
                        "nan_count": None,
                        "inf_count": None,
                        "read_error": str(exc)
                    })

            result["readable"] = True

    except Exception as exc:
        result["error"] = str(exc)

    return result


def inspect_json(json_path):
    """Проверяет JSON и анализирует структуру его полей."""

    result = {
        "readable": False,
        "root_type": None,
        "top_level_keys": 0,
        "field_count": 0,
        "empty_fields": 0,
        "fields": [],
        "error": None
    }

    if not json_path:
        result["error"] = "JSON path is missing"
        return result

    path = Path(str(json_path))

    if not path.is_file():
        result["error"] = "JSON file does not exist"
        return result

    try:
        with open(path, "r", encoding="utf-8") as file:
            data = json.load(file)

        result["readable"] = True
        result["root_type"] = type(data).__name__

        if isinstance(data, dict):
            result["top_level_keys"] = len(data)

        fields = flatten_json(data)

        result["fields"] = fields
        result["field_count"] = len(fields)

        result["empty_fields"] = sum(
            bool(item["is_empty"])
            for item in fields
        )

    except Exception as exc:
        result["error"] = str(exc)

    return result


# ============================================================
# 3. AUDIT ALL STRUCTURES
# ============================================================

npz_summary_records = []
json_summary_records = []

npz_array_records = []
json_field_records = []

audit_errors = []

for row_number, row in audit_df.iterrows():

    pdb_id = str(row["pdb_id"])

    npz_path = row["graph_file"]
    json_path = row["features_json"]

    print(f"\n[{row_number + 1}/{len(audit_df)}] {pdb_id}")

    # --------------------------------------------------------
    # NPZ
    # --------------------------------------------------------

    npz_result = inspect_npz(npz_path)

    npz_summary_records.append({
        "pdb_id": pdb_id,
        "npz_readable": npz_result["readable"],
        "array_count": npz_result["array_count"],
        "total_elements": npz_result["total_elements"],
        "empty_arrays": npz_result["empty_arrays"],
        "nan_count": npz_result["nan_count"],
        "inf_count": npz_result["inf_count"],
        "object_arrays": npz_result["object_arrays"],
        "npz_error": npz_result["error"]
    })

    for item in npz_result["arrays"]:

        npz_array_records.append({
            "pdb_id": pdb_id,
            **item
        })

    # --------------------------------------------------------
    # JSON
    # --------------------------------------------------------

    json_result = inspect_json(json_path)

    json_summary_records.append({
        "pdb_id": pdb_id,
        "json_readable": json_result["readable"],
        "root_type": json_result["root_type"],
        "top_level_keys": json_result["top_level_keys"],
        "field_count": json_result["field_count"],
        "empty_fields": json_result["empty_fields"],
        "json_error": json_result["error"]
    })

    for item in json_result["fields"]:

        json_field_records.append({
            "pdb_id": pdb_id,
            **item
        })

    # --------------------------------------------------------
    # ERRORS
    # --------------------------------------------------------

    if not npz_result["readable"]:
        audit_errors.append({
            "pdb_id": pdb_id,
            "file_type": "NPZ",
            "error": npz_result["error"]
        })

    if not json_result["readable"]:
        audit_errors.append({
            "pdb_id": pdb_id,
            "file_type": "JSON",
            "error": json_result["error"]
        })

    print(
        f"  NPZ: readable={npz_result['readable']}, "
        f"arrays={npz_result['array_count']}, "
        f"elements={npz_result['total_elements']}"
    )

    print(
        f"  JSON: readable={json_result['readable']}, "
        f"fields={json_result['field_count']}, "
        f"empty={json_result['empty_fields']}"
    )


# ============================================================
# 4. CREATE DATAFRAMES
# ============================================================

df_npz_audit = pd.DataFrame(npz_summary_records)
df_json_audit = pd.DataFrame(json_summary_records)

df_npz_arrays = pd.DataFrame(npz_array_records)
df_json_fields = pd.DataFrame(json_field_records)

df_audit_errors = pd.DataFrame(
    audit_errors,
    columns=["pdb_id", "file_type", "error"]
)


# ============================================================
# 5. COMPARE JSON FIELD PRESENCE
# ============================================================

if not df_json_fields.empty:

    df_json_field_presence = (
        df_json_fields[
            ["pdb_id", "json_path", "value_type"]
        ]
        .drop_duplicates()
        .pivot_table(
            index="json_path",
            columns="pdb_id",
            values="value_type",
            aggfunc="first"
        )
    )

    df_json_field_presence["structures_with_field"] = (
        df_json_field_presence.notna().sum(axis=1)
    )

    df_json_field_presence["coverage_percent"] = (
        df_json_field_presence["structures_with_field"]
        / len(audit_df)
        * 100
    )

    df_json_field_presence = (
        df_json_field_presence
        .sort_values(
            "coverage_percent",
            ascending=True
        )
    )

else:

    df_json_field_presence = pd.DataFrame()


# ============================================================
# 6. CREATE OVERALL AUDIT SUMMARY
# ============================================================

npz_readable_count = int(
    df_npz_audit["npz_readable"].sum()
)

json_readable_count = int(
    df_json_audit["json_readable"].sum()
)

npz_with_nan = int(
    (df_npz_audit["nan_count"] > 0).sum()
)

npz_with_inf = int(
    (df_npz_audit["inf_count"] > 0).sum()
)

npz_with_empty_arrays = int(
    (df_npz_audit["empty_arrays"] > 0).sum()
)

json_with_empty_fields = int(
    (df_json_audit["empty_fields"] > 0).sum()
)

fully_readable = int(
    (
        df_npz_audit["npz_readable"]
        & df_json_audit["json_readable"]
    ).sum()
)

print("\n" + "=" * 75)
print("CONTENT AUDIT SUMMARY")
print("=" * 75)

print(f"Total structures: {len(audit_df)}")

print(
    f"Readable NPZ: {npz_readable_count}/"
    f"{len(audit_df)}"
)

print(
    f"Readable JSON: {json_readable_count}/"
    f"{len(audit_df)}"
)

print(
    f"Both files readable: {fully_readable}/"
    f"{len(audit_df)}"
)

print("-" * 75)

print(f"NPZ with NaN: {npz_with_nan}")
print(f"NPZ with Inf: {npz_with_inf}")
print(f"NPZ with empty arrays: {npz_with_empty_arrays}")
print(f"JSON with empty fields: {json_with_empty_fields}")
print(f"Read errors: {len(df_audit_errors)}")

print("=" * 75)


# ============================================================
# 7. DISPLAY REPORTS
# ============================================================

print("\n1. NPZ SUMMARY")
display(
    df_npz_audit.sort_values("pdb_id").reset_index(drop=True)
)

print("\n2. JSON SUMMARY")
display(
    df_json_audit.sort_values("pdb_id").reset_index(drop=True)
)

print("\n3. NPZ ARRAY DETAILS")
display(
    df_npz_arrays.sort_values(
        ["pdb_id", "array_name"]
    ).reset_index(drop=True)
)

print("\n4. JSON FIELD COVERAGE")
if not df_json_field_presence.empty:
    display(df_json_field_presence)
else:
    print("JSON fields were not found.")

print("\n5. READ ERRORS")
if not df_audit_errors.empty:
    display(df_audit_errors)
else:
    print("No NPZ or JSON read errors.")


# ============================================================
# 8. SAVE AUDIT REPORTS
# ============================================================

if "BATCH_RUN_DIR" not in globals():
    raise RuntimeError(
        "BATCH_RUN_DIR не найден. "
        "Проверьте выполнение предыдущих ячеек."
    )

BATCH_RUN_DIR = Path(BATCH_RUN_DIR)
BATCH_RUN_DIR.mkdir(parents=True, exist_ok=True)

audit_files = {
    "content_audit_npz_summary.csv": df_npz_audit,
    "content_audit_json_summary.csv": df_json_audit,
    "content_audit_npz_arrays.csv": df_npz_arrays,
    "content_audit_json_fields.csv": df_json_fields,
    "content_audit_json_field_coverage.csv": (
        df_json_field_presence.reset_index()
        if not df_json_field_presence.empty
        else pd.DataFrame()
    ),
    "content_audit_errors.csv": df_audit_errors
}

for filename, dataframe in audit_files.items():

    output_path = BATCH_RUN_DIR / filename

    dataframe.to_csv(
        output_path,
        index=False,
        encoding="utf-8-sig"
    )

    print(f"Saved: {output_path}")


# ============================================================
# 9. FINAL STATUS
# ============================================================

print("\n" + "=" * 75)
print("CELL 8 COMPLETED")
print("=" * 75)

print(f"Structures audited: {len(audit_df)}")
print(f"Both files readable: {fully_readable}")
print(f"Read errors: {len(df_audit_errors)}")

print(
    "Report directory:",
    BATCH_RUN_DIR
)

print(
    "Finished:",
    datetime.now().isoformat(timespec="seconds")
)

print("=" * 75)


CELL 8. CONTENT AUDIT OF NPZ AND JSON OUTPUTS
Structures to audit: 100
Started: 2026-11-07T15:07:41

[1/100] 7WTU
  NPZ: readable=True, arrays=19, elements=6161290
  JSON: readable=True, fields=806, empty=41

[2/100] 7W37
  NPZ: readable=True, arrays=19, elements=14147078
  JSON: readable=True, fields=849, empty=33

[3/100] 7Q0G
  NPZ: readable=True, arrays=19, elements=1098756
  JSON: readable=True, fields=798, empty=42

[4/100] 7VYS
  NPZ: readable=True, arrays=19, elements=5211388
  JSON: readable=True, fields=794, empty=42

[5/100] 7X8R
  NPZ: readable=True, arrays=19, elements=1160612
  JSON: readable=True, fields=787, empty=43

[6/100] 7QB5
  NPZ: readable=True, arrays=19, elements=899925
  JSON: readable=True, fields=832, empty=33

[7/100] 7Z0O
  NPZ: readable=True, arrays=19, elements=1372516
  JSON: readable=True, fields=761, empty=47

[8/100] 7XCR
  NPZ: readable=True, arrays=19, elements=1242602
  JSON: readable=True, fields=781, empty=43

[9/100] 12TB
  NPZ: readable=True, 

,pdb_id,npz_readable,array_count,total_elements,empty_arrays,nan_count,inf_count,object_arrays,npz_error
0,11PH,True,19,1236600,0,0,0,0,None
1,12TB,True,19,232923,0,0,0,0,None
2,21ZV,True,19,2939334,0,0,0,0,None
3,21ZW,True,19,2365073,0,0,0,0,None
4,22IQ,True,19,1025844,0,0,0,0,None
...,...,...,...,...,...,...,...,...,...
95,7Z0O,True,19,1372516,0,0,0,0,None
96,7Z13,True,19,13848499,0,0,0,0,None
97,7Z16,True,19,3171987,0,0,0,0,None
98,7Z38,True,19,1912789,0,0,0,0,None



2. JSON SUMMARY


,pdb_id,json_readable,root_type,top_level_keys,field_count,empty_fields,json_error
0,11PH,True,dict,9,798,41,None
1,12TB,True,dict,9,803,40,None
2,21ZV,True,dict,9,783,46,None
3,21ZW,True,dict,9,835,31,None
4,22IQ,True,dict,9,755,47,None
...,...,...,...,...,...,...,...
95,7Z0O,True,dict,9,761,47,None
96,7Z13,True,dict,9,824,37,None
97,7Z16,True,dict,9,822,37,None
98,7Z38,True,dict,9,803,42,None



3. NPZ ARRAY DETAILS


,pdb_id,array_name,shape,dtype,elements,is_empty,is_numeric,nan_count,inf_count,read_error
0,11PH,X_atom,"(9283, 26)",float32,241358,False,True,0,0,None
1,11PH,X_global,"(122,)",float32,122,False,True,0,0,None
2,11PH,X_residue,"(1161, 23)",float32,26703,False,True,0,0,None
3,11PH,atom_edge_feature_names,"(2,)",<U12,2,False,False,0,0,None
4,11PH,atom_feature_names,"(26,)",<U23,26,False,False,0,0,None
...,...,...,...,...,...,...,...,...,...,...
1895,7Z6Q,pos_atom,"(31149, 3)",float32,93447,False,True,0,0,None
1896,7Z6Q,pos_residue,"(3956, 3)",float32,11868,False,True,0,0,None
1897,7Z6Q,residue_edge_feature_names,"(3,)",<U23,3,False,False,0,0,None
1898,7Z6Q,residue_feature_names,"(23,)",<U23,23,False,False,0,0,None



4. JSON FIELD COVERAGE


pdb_id,11PH,12TB,21ZV,21ZW,22IQ,3NUL,3NV0,3NWS,3NY7,3NZJ,...,7YE6,7YI0,7YMS,7Z0O,7Z13,7Z16,7Z38,7Z6Q,structures_with_field,coverage_percent
json_path,,,,,,,,,,,,,,,,,,,,,
global_features.metal_mean_distance_CU_A,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,1.0
global_features.metal_contact_record_count_CU,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,1.0
table_statistics.global_features.column_names[158],NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,1.0
table_statistics.global_features.column_names[157],NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,1.0
table_statistics.global_features.column_names[159],NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
validation.source_atom_row_count_preserved,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,...,bool,bool,bool,bool,bool,bool,bool,bool,100,100.0
validation.source_residue_row_count_preserved,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,...,bool,bool,bool,bool,bool,bool,bool,bool,100,100.0
validation.unobserved_atom_features_file_exists,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,...,bool,bool,bool,bool,bool,bool,bool,bool,100,100.0



5. READ ERRORS
No NPZ or JSON read errors.
Saved: C:\apex\APEX\МИФИ\СНО\medic\Protein_Pocket_Dataset\07_structure_analysis\batch_runs\protein_batch_v1\content_audit_npz_summary.csv
Saved: C:\apex\APEX\МИФИ\СНО\medic\Protein_Pocket_Dataset\07_structure_analysis\batch_runs\protein_batch_v1\content_audit_json_summary.csv
Saved: C:\apex\APEX\МИФИ\СНО\medic\Protein_Pocket_Dataset\07_structure_analysis\batch_runs\protein_batch_v1\content_audit_npz_arrays.csv
Saved: C:\apex\APEX\МИФИ\СНО\medic\Protein_Pocket_Dataset\07_structure_analysis\batch_runs\protein_batch_v1\content_audit_json_fields.csv
Saved: C:\apex\APEX\МИФИ\СНО\medic\Protein_Pocket_Dataset\07_structure_analysis\batch_runs\protein_batch_v1\content_audit_json_field_coverage.csv
Saved: C:\apex\APEX\МИФИ\СНО\medic\Protein_Pocket_Dataset\07_structure_analysis\batch_runs\protein_batch_v1\content_audit_errors.csv

CELL 8 COMPLETED
Structures audited: 100
Both files readable: 100
Read errors: 0
Report directory: C:\apex\APEX\МИФИ\СНО\med

In [10]:

# ============================================================
# CELL 9. DEEP STRUCTURAL CONSISTENCY AUDIT
# ============================================================
# Назначение:
# 1. Сравнить схемы NPZ между всеми структурами.
# 2. Проверить соответствие матриц признаков и feature_names.
# 3. Проверить согласованность числа атомов, остатков и координат.
# 4. Проанализировать пустые поля JSON.
# 5. Проверить единообразие верхнеуровневой структуры JSON.
# 6. Выявить отсутствующие и редкие поля.
#
# ВАЖНО:
# - Только чтение исходных файлов.
# - NPZ и JSON не изменяются.
# - CELL 17 и CELL 18 не запускаются.
# ============================================================

import json
import re
import numpy as np
import pandas as pd

from pathlib import Path
from collections import defaultdict, Counter
from datetime import datetime


# ============================================================
# 1. INPUT CHECK
# ============================================================

print("=" * 75)
print("CELL 9. DEEP STRUCTURAL CONSISTENCY AUDIT")
print("=" * 75)

if "df_batch_qc" not in globals():
    raise RuntimeError("Сначала выполните CELL 6.")

required_columns = {
    "pdb_id",
    "graph_file",
    "features_json"
}

missing = required_columns - set(df_batch_qc.columns)

if missing:
    raise RuntimeError(f"Отсутствуют столбцы: {missing}")

audit_input = df_batch_qc.copy()

print("Structures:", len(audit_input))
print("Started:", datetime.now().isoformat(timespec="seconds"))


# ============================================================
# 2. RESULT CONTAINERS
# ============================================================

npz_schema_records = []
npz_consistency_records = []

json_top_records = []
json_empty_records = []
json_field_records = []

global_array_names = set()
global_top_keys = set()


# ============================================================
# 3. HELPER FUNCTIONS
# ============================================================

def normalize_json_path(path):
    """Заменяет индексы списков на общий шаблон."""

    return re.sub(r"\[\d+\]", "[*]", str(path))


def flatten_json_audit(data, prefix=""):
    """Рекурсивный анализ JSON без сохранения значений."""

    records = []

    if isinstance(data, dict):

        for key, value in data.items():

            current = (
                f"{prefix}.{key}"
                if prefix else str(key)
            )

            if isinstance(value, dict):

                records.append({
                    "json_path": current,
                    "normalized_path": normalize_json_path(current),
                    "value_type": "dict",
                    "is_empty": len(value) == 0,
                    "is_null": False
                })

                records.extend(
                    flatten_json_audit(value, current)
                )

            elif isinstance(value, list):

                records.append({
                    "json_path": current,
                    "normalized_path": normalize_json_path(current),
                    "value_type": "list",
                    "is_empty": len(value) == 0,
                    "is_null": False
                })

                records.extend(
                    flatten_json_audit(value, current)
                )

            else:

                records.append({
                    "json_path": current,
                    "normalized_path": normalize_json_path(current),
                    "value_type": type(value).__name__,
                    "is_empty": (
                        isinstance(value, str)
                        and len(value.strip()) == 0
                    ),
                    "is_null": value is None
                })

    elif isinstance(data, list):

        for index, value in enumerate(data):

            current = f"{prefix}[{index}]"

            if isinstance(value, (dict, list)):
                records.extend(
                    flatten_json_audit(value, current)
                )
            else:
                records.append({
                    "json_path": current,
                    "normalized_path": normalize_json_path(current),
                    "value_type": type(value).__name__,
                    "is_empty": (
                        isinstance(value, str)
                        and len(value.strip()) == 0
                    ),
                    "is_null": value is None
                })

    return records


def get_array_shape(archive, name):
    """Безопасно получает форму массива."""

    try:
        return tuple(archive[name].shape)
    except Exception:
        return None


def get_array_size(archive, name):
    """Безопасно получает число элементов."""

    try:
        return int(archive[name].size)
    except Exception:
        return None


# ============================================================
# 4. NPZ SCHEMA AND DIMENSION CHECKS
# ============================================================

for position, row in enumerate(
    audit_input.itertuples(index=False),
    start=1
):

    pdb_id = str(row.pdb_id)
    npz_path = Path(str(row.graph_file))

    print(f"[{position}/{len(audit_input)}] {pdb_id}")

    if not npz_path.is_file():
        npz_consistency_records.append({
            "pdb_id": pdb_id,
            "check": "NPZ_EXISTS",
            "status": "FAIL",
            "details": "NPZ file not found"
        })
        continue

    try:

        with np.load(npz_path, allow_pickle=False) as archive:

            names = set(archive.files)
            global_array_names.update(names)

            # -----------------------------------------------
            # ARRAY SCHEMA
            # -----------------------------------------------

            for name in archive.files:

                array = archive[name]

                npz_schema_records.append({
                    "pdb_id": pdb_id,
                    "array_name": name,
                    "shape": str(tuple(array.shape)),
                    "ndim": array.ndim,
                    "dtype": str(array.dtype),
                    "elements": int(array.size)
                })

            # -----------------------------------------------
            # CHECK FEATURE MATRIX DIMENSIONS
            # -----------------------------------------------

            dimension_pairs = [
                ("X_atom", "atom_feature_names"),
                ("X_residue", "residue_feature_names"),
                ("X_global", "global_feature_names"),
                ("X_atom_edge", "atom_edge_feature_names"),
                ("X_residue_edge", "residue_edge_feature_names")
            ]

            for matrix_name, feature_name in dimension_pairs:

                if matrix_name not in names:
                    continue

                if feature_name not in names:
                    npz_consistency_records.append({
                        "pdb_id": pdb_id,
                        "check": f"{matrix_name}_FEATURE_NAMES",
                        "status": "WARNING",
                        "details": (
                            f"{feature_name} not found"
                        )
                    })
                    continue

                matrix = archive[matrix_name]
                feature_names = archive[feature_name]

                if matrix.ndim == 0:
                    status = "FAIL"
                    details = "Feature matrix is scalar"

                elif matrix.shape[-1] != feature_names.size:
                    status = "FAIL"
                    details = (
                        f"Matrix columns={matrix.shape[-1]}, "
                        f"feature names={feature_names.size}"
                    )

                else:
                    status = "PASS"
                    details = (
                        f"Columns={matrix.shape[-1]}, "
                        f"feature names={feature_names.size}"
                    )

                npz_consistency_records.append({
                    "pdb_id": pdb_id,
                    "check": f"{matrix_name}_FEATURE_NAMES",
                    "status": status,
                    "details": details
                })

            # -----------------------------------------------
            # CHECK ATOM ROWS VS ATOM COORDINATES
            # -----------------------------------------------

            if (
                "X_atom" in names
                and "pos_atom" in names
            ):

                x = archive["X_atom"]
                pos = archive["pos_atom"]

                if x.ndim >= 1 and pos.ndim >= 1:
                    match = x.shape[0] == pos.shape[0]

                    npz_consistency_records.append({
                        "pdb_id": pdb_id,
                        "check": "ATOM_FEATURES_VS_COORDINATES",
                        "status": "PASS" if match else "FAIL",
                        "details": (
                            f"X_atom rows={x.shape[0]}, "
                            f"pos_atom rows={pos.shape[0]}"
                        )
                    })

            # -----------------------------------------------
            # CHECK RESIDUE ROWS VS RESIDUE COORDINATES
            # -----------------------------------------------

            if (
                "X_residue" in names
                and "pos_residue" in names
            ):

                x = archive["X_residue"]
                pos = archive["pos_residue"]

                if x.ndim >= 1 and pos.ndim >= 1:
                    match = x.shape[0] == pos.shape[0]

                    npz_consistency_records.append({
                        "pdb_id": pdb_id,
                        "check": "RESIDUE_FEATURES_VS_COORDINATES",
                        "status": "PASS" if match else "FAIL",
                        "details": (
                            f"X_residue rows={x.shape[0]}, "
                            f"pos_residue rows={pos.shape[0]}"
                        )
                    })

            # -----------------------------------------------
            # CHECK COORDINATE DIMENSIONS
            # -----------------------------------------------

            for name in ("pos_atom", "pos_residue"):

                if name not in names:
                    continue

                array = archive[name]

                valid_shape = (
                    array.ndim == 2
                    and array.shape[1] == 3
                )

                npz_consistency_records.append({
                    "pdb_id": pdb_id,
                    "check": f"{name}_DIMENSION",
                    "status": "PASS" if valid_shape else "FAIL",
                    "details": f"Shape={array.shape}"
                })

            # -----------------------------------------------
            # CHECK RESIDUE NODE IDS
            # -----------------------------------------------

            if "residue_node_ids" in names:

                ids = archive["residue_node_ids"]

                valid = (
                    ids.ndim == 1
                    and (
                        "X_residue" not in names
                        or ids.size == archive["X_residue"].shape[0]
                    )
                )

                npz_consistency_records.append({
                    "pdb_id": pdb_id,
                    "check": "RESIDUE_NODE_IDS",
                    "status": "PASS" if valid else "FAIL",
                    "details": (
                        f"IDs={ids.size}, "
                        f"residues="
                        f"{archive['X_residue'].shape[0] if 'X_residue' in names else 'N/A'}"
                    )
                })

    except Exception as exc:

        npz_consistency_records.append({
            "pdb_id": pdb_id,
            "check": "NPZ_READ",
            "status": "FAIL",
            "details": str(exc)
        })


# ============================================================
# 5. JSON SCHEMA AND EMPTY FIELD ANALYSIS
# ============================================================

for position, row in enumerate(
    audit_input.itertuples(index=False),
    start=1
):

    pdb_id = str(row.pdb_id)
    json_path = Path(str(row.features_json))

    if not json_path.is_file():
        continue

    try:

        with open(json_path, "r", encoding="utf-8") as file:
            data = json.load(file)

        if isinstance(data, dict):

            keys = set(data.keys())
            global_top_keys.update(keys)

            json_top_records.append({
                "pdb_id": pdb_id,
                "top_level_key_count": len(keys),
                "top_level_keys": "|".join(sorted(keys))
            })

        else:

            json_top_records.append({
                "pdb_id": pdb_id,
                "top_level_key_count": None,
                "top_level_keys": None
            })

        fields = flatten_json_audit(data)

        for item in fields:

            record = {
                "pdb_id": pdb_id,
                **item
            }

            json_field_records.append(record)

            if item["is_empty"] or item["is_null"]:

                json_empty_records.append({
                    "pdb_id": pdb_id,
                    "json_path": item["json_path"],
                    "normalized_path": item["normalized_path"],
                    "value_type": item["value_type"],
                    "is_empty": item["is_empty"],
                    "is_null": item["is_null"]
                })

    except Exception as exc:

        json_top_records.append({
            "pdb_id": pdb_id,
            "top_level_key_count": None,
            "top_level_keys": f"ERROR: {exc}"
        })


# ============================================================
# 6. DATAFRAMES
# ============================================================

df_npz_schema = pd.DataFrame(npz_schema_records)

df_npz_consistency = pd.DataFrame(
    npz_consistency_records
)

df_json_top = pd.DataFrame(json_top_records)

df_json_empty = pd.DataFrame(json_empty_records)

df_json_all_fields = pd.DataFrame(json_field_records)


# ============================================================
# 7. NPZ ARRAY NAME COVERAGE
# ============================================================

if not df_npz_schema.empty:

    df_npz_array_coverage = (
        df_npz_schema[
            ["pdb_id", "array_name"]
        ]
        .drop_duplicates()
        .groupby("array_name")
        .agg(
            structures_with_array=("pdb_id", "nunique")
        )
        .reset_index()
    )

    df_npz_array_coverage["coverage_percent"] = (
        df_npz_array_coverage["structures_with_array"]
        / len(audit_input)
        * 100
    )

    df_npz_shape_variation = (
        df_npz_schema
        .groupby("array_name")
        .agg(
            structures=("pdb_id", "nunique"),
            different_shapes=("shape", "nunique"),
            different_dtypes=("dtype", "nunique"),
            example_shapes=(
                "shape",
                lambda x: " | ".join(
                    sorted(set(x.astype(str)))[:8]
                )
            )
        )
        .reset_index()
    )

else:

    df_npz_array_coverage = pd.DataFrame()
    df_npz_shape_variation = pd.DataFrame()


# ============================================================
# 8. JSON TOP-LEVEL KEY COVERAGE
# ============================================================

json_top_coverage_records = []

for key in sorted(global_top_keys):

    present = 0

    for row in df_json_top.itertuples(index=False):

        keys = (
            set(str(row.top_level_keys).split("|"))
            if row.top_level_keys
            else set()
        )

        if key in keys:
            present += 1

    json_top_coverage_records.append({
        "top_level_key": key,
        "structures_with_key": present,
        "coverage_percent": (
            present / len(audit_input) * 100
        )
    })

df_json_top_coverage = pd.DataFrame(
    json_top_coverage_records
)


# ============================================================
# 9. JSON EMPTY FIELD FREQUENCY
# ============================================================

if not df_json_empty.empty:

    df_json_empty_frequency = (
        df_json_empty
        .groupby(
            ["normalized_path", "value_type"],
            dropna=False
        )
        .agg(
            affected_structures=("pdb_id", "nunique"),
            occurrences=("pdb_id", "size"),
            example_pdbs=(
                "pdb_id",
                lambda x: ", ".join(
                    sorted(set(x.astype(str)))[:10]
                )
            ),
            has_null=("is_null", "max"),
            has_empty=("is_empty", "max")
        )
        .reset_index()
        .sort_values(
            ["affected_structures", "occurrences"],
            ascending=False
        )
    )

else:

    df_json_empty_frequency = pd.DataFrame()


# ============================================================
# 10. JSON FIELD COVERAGE
# ============================================================

if not df_json_all_fields.empty:

    df_json_field_frequency = (
        df_json_all_fields
        .groupby(
            ["normalized_path", "value_type"],
            dropna=False
        )
        .agg(
            structures_with_field=("pdb_id", "nunique"),
            occurrences=("pdb_id", "size")
        )
        .reset_index()
    )

    df_json_field_frequency["coverage_percent"] = (
        df_json_field_frequency["structures_with_field"]
        / len(audit_input)
        * 100
    )

    df_json_field_frequency = (
        df_json_field_frequency
        .sort_values(
            ["coverage_percent", "normalized_path"],
            ascending=[True, True]
        )
    )

else:

    df_json_field_frequency = pd.DataFrame()


# ============================================================
# 11. SUMMARY
# ============================================================

print("\n" + "=" * 75)
print("NPZ STRUCTURAL CONSISTENCY")
print("=" * 75)

if not df_npz_consistency.empty:

    print(
        df_npz_consistency["status"]
        .value_counts(dropna=False)
        .to_string()
    )

    failed_checks = df_npz_consistency[
        df_npz_consistency["status"] == "FAIL"
    ]

    warning_checks = df_npz_consistency[
        df_npz_consistency["status"] == "WARNING"
    ]

    print("\nFailed checks:", len(failed_checks))
    print("Warnings:", len(warning_checks))

else:

    print("No NPZ consistency checks were produced.")


print("\n" + "=" * 75)
print("NPZ ARRAY SCHEMA")
print("=" * 75)

print("Unique array names:", len(global_array_names))

if not df_npz_array_coverage.empty:
    display(
        df_npz_array_coverage.sort_values(
            "coverage_percent"
        )
    )

print("\nARRAY SHAPE VARIATION")
if not df_npz_shape_variation.empty:
    display(df_npz_shape_variation)


print("\n" + "=" * 75)
print("JSON TOP-LEVEL SCHEMA")
print("=" * 75)

display(df_json_top_coverage)


print("\n" + "=" * 75)
print("MOST FREQUENT EMPTY JSON FIELDS")
print("=" * 75)

if not df_json_empty_frequency.empty:
    display(df_json_empty_frequency.head(50))
else:
    print("No empty or null JSON fields found.")


print("\n" + "=" * 75)
print("JSON FIELD COVERAGE: RARE FIELDS")
print("=" * 75)

if not df_json_field_frequency.empty:
    display(df_json_field_frequency.head(50))
else:
    print("No JSON fields found.")


# ============================================================
# 12. DISPLAY FAILED NPZ CHECKS
# ============================================================

print("\n" + "=" * 75)
print("FAILED NPZ CONSISTENCY CHECKS")
print("=" * 75)

if not df_npz_consistency.empty:

    failed = df_npz_consistency[
        df_npz_consistency["status"] == "FAIL"
    ]

    if not failed.empty:
        display(failed)
    else:
        print("No failed NPZ consistency checks.")

else:
    print("No consistency checks available.")


# ============================================================
# 13. SAVE REPORTS
# ============================================================

if "BATCH_RUN_DIR" not in globals():
    raise RuntimeError("BATCH_RUN_DIR не найден.")

BATCH_RUN_DIR = Path(BATCH_RUN_DIR)
BATCH_RUN_DIR.mkdir(parents=True, exist_ok=True)

reports = {
    "deep_audit_npz_schema.csv": df_npz_schema,
    "deep_audit_npz_consistency.csv": df_npz_consistency,
    "deep_audit_npz_array_coverage.csv": df_npz_array_coverage,
    "deep_audit_npz_shape_variation.csv": df_npz_shape_variation,
    "deep_audit_json_top_level.csv": df_json_top,
    "deep_audit_json_top_coverage.csv": df_json_top_coverage,
    "deep_audit_json_empty_fields.csv": df_json_empty,
    "deep_audit_json_empty_frequency.csv": df_json_empty_frequency,
    "deep_audit_json_field_coverage.csv": df_json_field_frequency
}

for filename, dataframe in reports.items():

    output_path = BATCH_RUN_DIR / filename

    dataframe.to_csv(
        output_path,
        index=False,
        encoding="utf-8-sig"
    )

    print("Saved:", output_path)


# ============================================================
# 14. FINAL STATUS
# ============================================================

print("\n" + "=" * 75)
print("CELL 9 COMPLETED")
print("=" * 75)

print("Structures audited:", len(audit_input))
print("NPZ array names:", len(global_array_names))

if not df_npz_consistency.empty:
    print(
        "Failed NPZ checks:",
        int(
            (
                df_npz_consistency["status"] == "FAIL"
            ).sum()
        )
    )

print(
    "JSON empty/null records:",
    len(df_json_empty)
)

print("Report directory:", BATCH_RUN_DIR)
print("Finished:", datetime.now().isoformat(timespec="seconds"))

print("=" * 75)


CELL 9. DEEP STRUCTURAL CONSISTENCY AUDIT
Structures: 100
Started: 2026-11-07T15:10:49
[1/100] 7WTU
[2/100] 7W37
[3/100] 7Q0G
[4/100] 7VYS
[5/100] 7X8R
[6/100] 7QB5
[7/100] 7Z0O
[8/100] 7XCR
[9/100] 12TB
[10/100] 3PUW
[11/100] 7WTT
[12/100] 3O2R
[13/100] 3NV0
[14/100] 7Z16
[15/100] 7WSO
[16/100] 7PR4
[17/100] 7W38
[18/100] 3OFG
[19/100] 7XNX
[20/100] 3P1A
[21/100] 7W1Z
[22/100] 7OYK
[23/100] 7WIG
[24/100] 7PS6
[25/100] 7W2U
[26/100] 3NZJ
[27/100] 7XCT
[28/100] 7YMS
[29/100] 3NY7
[30/100] 3P0Y
[31/100] 7Y3G
[32/100] 7YE4
[33/100] 7P4R
[34/100] 7W5Z
[35/100] 7PCA
[36/100] 7WU2
[37/100] 3OS8
[38/100] 7YE6
[39/100] 3P3O
[40/100] 7Z13
[41/100] 7PEF
[42/100] 7XD0
[43/100] 7PU4
[44/100] 3OR7
[45/100] 7W2Y
[46/100] 3O0A
[47/100] 7XMT
[48/100] 3PC8
[49/100] 21ZV
[50/100] 3O1C
[51/100] 7Y7M
[52/100] 3PSN
[53/100] 7W4C
[54/100] 3NWS
[55/100] 7Z38
[56/100] 3POZ
[57/100] 7VZ4
[58/100] 3OZZ
[59/100] 7Y75
[60/100] 3P8V
[61/100] 7XCP
[62/100] 3O1X
[63/100] 7XW6
[64/100] 3NZL
[65/100] 7W4F
[66/100] 3OS

,array_name,structures_with_array,coverage_percent
0,X_atom,100,100.0
1,X_global,100,100.0
2,X_residue,100,100.0
3,atom_edge_feature_names,100,100.0
4,atom_feature_names,100,100.0
5,atom_node_ids,100,100.0
6,edge_attr_atom,100,100.0
7,edge_attr_residue,100,100.0
8,edge_index_atom,100,100.0
9,edge_index_residue,100,100.0



ARRAY SHAPE VARIATION


,array_name,structures,different_shapes,different_dtypes,example_shapes
0,X_atom,100,98,1,"(10116, 26) | (10265, 26) | (104832, 29) | (10..."
1,X_global,100,20,1,"(113,) | (114,) | (115,) | (116,) | (117,) | (..."
2,X_residue,100,90,1,"(1001, 23) | (1010, 23) | (1013, 23) | (1017, ..."
3,atom_edge_feature_names,100,1,1,"(2,)"
4,atom_feature_names,100,4,3,"(26,) | (27,) | (28,) | (29,)"
5,atom_node_ids,100,98,1,"(10116,) | (10265,) | (104832,) | (105837,) | ..."
6,edge_attr_atom,100,100,1,"(1042252, 2) | (106194, 2) | (108924, 2) | (10..."
7,edge_attr_residue,100,100,1,"(10416, 3) | (10484, 3) | (1056, 3) | (10662, ..."
8,edge_index_atom,100,100,1,"(2, 1042252) | (2, 106194) | (2, 108924) | (2,..."
9,edge_index_residue,100,100,1,"(2, 10416) | (2, 10484) | (2, 1056) | (2, 1066..."



JSON TOP-LEVEL SCHEMA


,top_level_key,structures_with_key,coverage_percent
0,analysis_notes,100,100.0
1,dataframe_sources,100,100.0
2,feature_definitions,100,100.0
3,geometry_features,100,100.0
4,global_features,100,100.0
5,metadata,100,100.0
6,quality_features,100,100.0
7,table_statistics,100,100.0
8,validation,100,100.0



MOST FREQUENT EMPTY JSON FIELDS


,normalized_path,value_type,affected_structures,occurrences,example_pdbs,has_null,has_empty
0,dataframe_sources.altloc_atoms,NoneType,100,100,"11PH, 12TB, 21ZV, 21ZW, 22IQ, 3NUL, 3NV0, 3NWS...",True,False
1,dataframe_sources.altloc_occupancy,NoneType,100,100,"11PH, 12TB, 21ZV, 21ZW, 22IQ, 3NUL, 3NV0, 3NWS...",True,False
2,dataframe_sources.atom_quality,NoneType,100,100,"11PH, 12TB, 21ZV, 21ZW, 22IQ, 3NUL, 3NV0, 3NWS...",True,False
3,dataframe_sources.chains,NoneType,100,100,"11PH, 12TB, 21ZV, 21ZW, 22IQ, 3NUL, 3NV0, 3NWS...",True,False
4,dataframe_sources.geometry,NoneType,100,100,"11PH, 12TB, 21ZV, 21ZW, 22IQ, 3NUL, 3NV0, 3NWS...",True,False
5,dataframe_sources.hydrogen_bonds,NoneType,100,100,"11PH, 12TB, 21ZV, 21ZW, 22IQ, 3NUL, 3NV0, 3NWS...",True,False
6,dataframe_sources.hydrophobic_contacts,NoneType,100,100,"11PH, 12TB, 21ZV, 21ZW, 22IQ, 3NUL, 3NV0, 3NWS...",True,False
7,dataframe_sources.invalid_occupancy_atoms,NoneType,100,100,"11PH, 12TB, 21ZV, 21ZW, 22IQ, 3NUL, 3NV0, 3NWS...",True,False
8,dataframe_sources.low_occupancy_atoms,NoneType,100,100,"11PH, 12TB, 21ZV, 21ZW, 22IQ, 3NUL, 3NV0, 3NWS...",True,False
9,dataframe_sources.missing_occupancy_atoms,NoneType,100,100,"11PH, 12TB, 21ZV, 21ZW, 22IQ, 3NUL, 3NV0, 3NWS...",True,False



JSON FIELD COVERAGE: RARE FIELDS


,normalized_path,value_type,structures_with_field,occurrences,coverage_percent
117,global_features.component_record_count_03P,int,1,1,1.0
118,global_features.component_record_count_1MA,int,1,1,1.0
119,global_features.component_record_count_1PW,int,1,1,1.0
120,global_features.component_record_count_2MG,int,1,1,1.0
122,global_features.component_record_count_4AC,int,1,1,1.0
123,global_features.component_record_count_5MC,int,1,1,1.0
124,global_features.component_record_count_5MU,int,1,1,1.0
125,global_features.component_record_count_6MZ,int,1,1,1.0
126,global_features.component_record_count_80M,int,1,1,1.0
128,global_features.component_record_count_970,int,1,1,1.0



FAILED NPZ CONSISTENCY CHECKS
No failed NPZ consistency checks.
Saved: C:\apex\APEX\МИФИ\СНО\medic\Protein_Pocket_Dataset\07_structure_analysis\batch_runs\protein_batch_v1\deep_audit_npz_schema.csv
Saved: C:\apex\APEX\МИФИ\СНО\medic\Protein_Pocket_Dataset\07_structure_analysis\batch_runs\protein_batch_v1\deep_audit_npz_consistency.csv
Saved: C:\apex\APEX\МИФИ\СНО\medic\Protein_Pocket_Dataset\07_structure_analysis\batch_runs\protein_batch_v1\deep_audit_npz_array_coverage.csv
Saved: C:\apex\APEX\МИФИ\СНО\medic\Protein_Pocket_Dataset\07_structure_analysis\batch_runs\protein_batch_v1\deep_audit_npz_shape_variation.csv
Saved: C:\apex\APEX\МИФИ\СНО\medic\Protein_Pocket_Dataset\07_structure_analysis\batch_runs\protein_batch_v1\deep_audit_json_top_level.csv
Saved: C:\apex\APEX\МИФИ\СНО\medic\Protein_Pocket_Dataset\07_structure_analysis\batch_runs\protein_batch_v1\deep_audit_json_top_coverage.csv
Saved: C:\apex\APEX\МИФИ\СНО\medic\Protein_Pocket_Dataset\07_structure_analysis\batch_runs\protein

In [11]:

# ============================================================
# CELL 10. FEATURE SCHEMA AND GRAPH INTEGRITY AUDIT
# ============================================================
# Назначение:
# 1. Сравнить названия и порядок признаков между всеми PDB.
# 2. Выявить различия в X_atom, X_residue и X_global.
# 3. Проверить дублирование названий признаков.
# 4. Проверить соответствие edge_index и edge_attr.
# 5. Проверить диапазоны индексов графовых рёбер.
# 6. Проверить длину масок и идентификаторов узлов.
# 7. Сформировать отчёт о совместимости графов.
#
# ВАЖНО:
# - Только чтение NPZ.
# - Исходные файлы не изменяются.
# - Никакого повторного анализа PDB.
# ============================================================

import numpy as np
import pandas as pd

from pathlib import Path
from collections import defaultdict, Counter
from datetime import datetime


# ============================================================
# 1. INPUT CHECK
# ============================================================

print("=" * 75)
print("CELL 10. FEATURE SCHEMA AND GRAPH INTEGRITY AUDIT")
print("=" * 75)

if "df_batch_qc" not in globals():
    raise RuntimeError("Сначала выполните CELL 6.")

required_columns = {"pdb_id", "graph_file"}

missing = required_columns - set(df_batch_qc.columns)

if missing:
    raise RuntimeError(f"Отсутствуют столбцы: {missing}")

audit_input = df_batch_qc.copy()

print("Structures:", len(audit_input))
print("Started:", datetime.now().isoformat(timespec="seconds"))


# ============================================================
# 2. FEATURE SCHEMA CONTAINERS
# ============================================================

feature_arrays = {
    "X_atom": "atom_feature_names",
    "X_residue": "residue_feature_names",
    "X_global": "global_feature_names"
}

feature_schema_records = []
feature_summary_records = []
graph_check_records = []

schemas_by_type = {
    name: defaultdict(list)
    for name in feature_arrays
}

feature_names_by_pdb = {
    name: {}
    for name in feature_arrays
}


# ============================================================
# 3. HELPERS
# ============================================================

def decode_feature_names(array):
    """Преобразует массив имён признаков в список строк."""

    result = []

    for value in np.asarray(array).reshape(-1):

        if isinstance(value, bytes):
            value = value.decode("utf-8", errors="replace")

        result.append(str(value).strip())

    return result


def add_check(pdb_id, check, status, details):
    graph_check_records.append({
        "pdb_id": pdb_id,
        "check": check,
        "status": status,
        "details": str(details)
    })


def check_edge_graph(
    archive,
    pdb_id,
    edge_index_name,
    edge_attr_name,
    node_count,
    graph_label
):
    """Проверяет рёбра и их признаки."""

    names = set(archive.files)

    if edge_index_name not in names:
        add_check(
            pdb_id, f"{graph_label}_EDGE_INDEX",
            "WARNING", "edge_index missing"
        )
        return

    edge_index = archive[edge_index_name]

    if (
        edge_index.ndim != 2
        or edge_index.shape[0] != 2
    ):
        add_check(
            pdb_id, f"{graph_label}_EDGE_INDEX_SHAPE",
            "FAIL", f"Unexpected shape: {edge_index.shape}"
        )
        return

    edge_count = edge_index.shape[1]

    add_check(
        pdb_id,
        f"{graph_label}_EDGE_INDEX_SHAPE",
        "PASS",
        f"Edges={edge_count}"
    )

    # Индексы узлов должны находиться в диапазоне [0, N).

    if edge_count > 0:

        min_index = int(edge_index.min())
        max_index = int(edge_index.max())

        valid_indices = (
            min_index >= 0
            and max_index < node_count
        )

        add_check(
            pdb_id,
            f"{graph_label}_EDGE_INDEX_BOUNDS",
            "PASS" if valid_indices else "FAIL",
            (
                f"min={min_index}, max={max_index}, "
                f"node_count={node_count}"
            )
        )

    else:

        add_check(
            pdb_id,
            f"{graph_label}_EDGE_INDEX_BOUNDS",
            "PASS",
            "Graph has no edges"
        )

    # Число строк edge_attr должно совпадать с числом рёбер.

    if edge_attr_name not in names:

        add_check(
            pdb_id,
            f"{graph_label}_EDGE_ATTR",
            "WARNING",
            "edge_attr missing"
        )

    else:

        edge_attr = archive[edge_attr_name]

        valid_attr = (
            edge_attr.ndim >= 1
            and edge_attr.shape[0] == edge_count
        )

        add_check(
            pdb_id,
            f"{graph_label}_EDGE_ATTR_ALIGNMENT",
            "PASS" if valid_attr else "FAIL",
            (
                f"edge_index edges={edge_count}, "
                f"edge_attr shape={edge_attr.shape}"
            )
        )


# ============================================================
# 4. AUDIT ALL NPZ FILES
# ============================================================

for position, row in enumerate(
    audit_input.itertuples(index=False),
    start=1
):

    pdb_id = str(row.pdb_id)
    npz_path = Path(str(row.graph_file))

    print(f"[{position}/{len(audit_input)}] {pdb_id}")

    if not npz_path.is_file():
        add_check(
            pdb_id, "NPZ_EXISTS", "FAIL", "File not found"
        )
        continue

    try:

        with np.load(npz_path, allow_pickle=False) as archive:

            names = set(archive.files)

            # ------------------------------------------------
            # 4.1 FEATURE SCHEMA
            # ------------------------------------------------

            for matrix_name, feature_name in feature_arrays.items():

                if (
                    matrix_name not in names
                    or feature_name not in names
                ):
                    add_check(
                        pdb_id,
                        f"{matrix_name}_SCHEMA",
                        "FAIL",
                        "Feature matrix or feature names missing"
                    )
                    continue

                matrix = archive[matrix_name]
                names_array = archive[feature_name]

                feature_names = decode_feature_names(
                    names_array
                )

                if matrix.ndim != 2:

                    add_check(
                        pdb_id,
                        f"{matrix_name}_NDIM",
                        "FAIL",
                        f"Expected 2D matrix, got {matrix.shape}"
                    )
                    continue

                if matrix.shape[1] != len(feature_names):

                    add_check(
                        pdb_id,
                        f"{matrix_name}_FEATURE_COUNT",
                        "FAIL",
                        (
                            f"Columns={matrix.shape[1]}, "
                            f"names={len(feature_names)}"
                        )
                    )

                else:

                    add_check(
                        pdb_id,
                        f"{matrix_name}_FEATURE_COUNT",
                        "PASS",
                        f"Features={len(feature_names)}"
                    )

                # Полная схема учитывает порядок признаков.

                schema_tuple = tuple(feature_names)

                schemas_by_type[matrix_name][
                    schema_tuple
                ].append(pdb_id)

                feature_names_by_pdb[matrix_name][
                    pdb_id
                ] = feature_names

                duplicate_names = [
                    name
                    for name, count in Counter(
                        feature_names
                    ).items()
                    if count > 1
                ]

                add_check(
                    pdb_id,
                    f"{matrix_name}_DUPLICATE_NAMES",
                    "FAIL" if duplicate_names else "PASS",
                    (
                        ", ".join(duplicate_names[:10])
                        if duplicate_names
                        else "No duplicate feature names"
                    )
                )

                feature_schema_records.append({
                    "pdb_id": pdb_id,
                    "matrix": matrix_name,
                    "rows": matrix.shape[0],
                    "columns": matrix.shape[1],
                    "feature_names_count": len(feature_names),
                    "schema_id": hash(schema_tuple),
                    "feature_names": " | ".join(feature_names)
                })

            # ------------------------------------------------
            # 4.2 ATOM GRAPH
            # ------------------------------------------------

            if "X_atom" in names:

                atom_count = archive["X_atom"].shape[0]

                check_edge_graph(
                    archive,
                    pdb_id,
                    "edge_index_atom",
                    "edge_attr_atom",
                    atom_count,
                    "ATOM"
                )

                for mask_name in (
                    "is_binding_site_atom",
                    "is_direct_contact_atom"
                ):

                    if mask_name not in names:
                        add_check(
                            pdb_id,
                            f"{mask_name}_LENGTH",
                            "WARNING",
                            "Mask missing"
                        )
                        continue

                    mask = archive[mask_name]

                    valid = (
                        mask.ndim == 1
                        and mask.size == atom_count
                    )

                    add_check(
                        pdb_id,
                        f"{mask_name}_LENGTH",
                        "PASS" if valid else "FAIL",
                        (
                            f"Mask size={mask.size}, "
                            f"atom count={atom_count}"
                        )
                    )

                if "atom_node_ids" in names:

                    ids = archive["atom_node_ids"]

                    valid = (
                        ids.ndim == 1
                        and ids.size == atom_count
                    )

                    add_check(
                        pdb_id,
                        "ATOM_NODE_IDS_LENGTH",
                        "PASS" if valid else "FAIL",
                        (
                            f"IDs={ids.size}, "
                            f"atom count={atom_count}"
                        )
                    )

            # ------------------------------------------------
            # 4.3 RESIDUE GRAPH
            # ------------------------------------------------

            if "X_residue" in names:

                residue_count = archive["X_residue"].shape[0]

                check_edge_graph(
                    archive,
                    pdb_id,
                    "edge_index_residue",
                    "edge_attr_residue",
                    residue_count,
                    "RESIDUE"
                )

                if "is_binding_site_residue" in names:

                    mask = archive["is_binding_site_residue"]

                    valid = (
                        mask.ndim == 1
                        and mask.size == residue_count
                    )

                    add_check(
                        pdb_id,
                        "BINDING_SITE_RESIDUE_MASK_LENGTH",
                        "PASS" if valid else "FAIL",
                        (
                            f"Mask size={mask.size}, "
                            f"residue count={residue_count}"
                        )
                    )

                if "residue_node_ids" in names:

                    ids = archive["residue_node_ids"]

                    valid = (
                        ids.ndim == 1
                        and ids.size == residue_count
                    )

                    add_check(
                        pdb_id,
                        "RESIDUE_NODE_IDS_LENGTH",
                        "PASS" if valid else "FAIL",
                        (
                            f"IDs={ids.size}, "
                            f"residue count={residue_count}"
                        )
                    )

    except Exception as exc:

        add_check(
            pdb_id,
            "NPZ_AUDIT",
            "FAIL",
            str(exc)
        )


# ============================================================
# 5. COMPARE FEATURE SCHEMAS
# ============================================================

for matrix_name, schema_dict in schemas_by_type.items():

    for schema_number, (schema, pdb_ids) in enumerate(
        schema_dict.items(),
        start=1
    ):

        feature_summary_records.append({
            "matrix": matrix_name,
            "schema_number": schema_number,
            "structures_count": len(pdb_ids),
            "pdb_ids": ", ".join(pdb_ids),
            "feature_count": len(schema),
            "feature_names": " | ".join(schema)
        })


df_feature_schemas = pd.DataFrame(
    feature_schema_records
)

df_feature_summary = pd.DataFrame(
    feature_summary_records
)

df_graph_checks = pd.DataFrame(
    graph_check_records
)


# ============================================================
# 6. FEATURE NAME DIFFERENCES
# ============================================================

feature_difference_records = []

for matrix_name, pdb_map in feature_names_by_pdb.items():

    all_names = set()

    for names_list in pdb_map.values():
        all_names.update(names_list)

    for pdb_id, names_list in pdb_map.items():

        current_set = set(names_list)

        missing_names = sorted(all_names - current_set)
        extra_names = sorted(current_set - all_names)

        feature_difference_records.append({
            "pdb_id": pdb_id,
            "matrix": matrix_name,
            "feature_count": len(names_list),
            "missing_feature_count": len(missing_names),
            "extra_feature_count": len(extra_names),
            "missing_features": " | ".join(missing_names),
            "extra_features": " | ".join(extra_names)
        })


df_feature_differences = pd.DataFrame(
    feature_difference_records
)


# ============================================================
# 7. SCHEMA ORDER COMPARISON
# ============================================================

order_records = []

for matrix_name, pdb_map in feature_names_by_pdb.items():

    if not pdb_map:
        continue

    reference_pdb = next(iter(pdb_map))
    reference_names = pdb_map[reference_pdb]

    for pdb_id, names_list in pdb_map.items():

        same_order = names_list == reference_names

        same_set = set(names_list) == set(reference_names)

        order_records.append({
            "matrix": matrix_name,
            "reference_pdb": reference_pdb,
            "pdb_id": pdb_id,
            "same_feature_set": same_set,
            "same_feature_order": same_order
        })


df_feature_order = pd.DataFrame(order_records)


# ============================================================
# 8. GRAPH CHECK SUMMARY
# ============================================================

print("\n" + "=" * 75)
print("GRAPH INTEGRITY SUMMARY")
print("=" * 75)

if not df_graph_checks.empty:

    display(
        df_graph_checks["status"]
        .value_counts()
        .rename_axis("status")
        .reset_index(name="checks")
    )

    failed = df_graph_checks[
        df_graph_checks["status"] == "FAIL"
    ]

    warnings = df_graph_checks[
        df_graph_checks["status"] == "WARNING"
    ]

    print("Failed checks:", len(failed))
    print("Warnings:", len(warnings))

else:

    print("No graph checks produced.")


# ============================================================
# 9. FEATURE SCHEMA SUMMARY
# ============================================================

print("\n" + "=" * 75)
print("FEATURE SCHEMA VARIANTS")
print("=" * 75)

if not df_feature_summary.empty:
    display(
        df_feature_summary[
            [
                "matrix",
                "schema_number",
                "structures_count",
                "feature_count",
                "pdb_ids"
            ]
        ]
    )


# ============================================================
# 10. FEATURE ORDER SUMMARY
# ============================================================

print("\n" + "=" * 75)
print("FEATURE ORDER COMPARISON")
print("=" * 75)

if not df_feature_order.empty:

    for matrix_name in feature_arrays:

        subset = df_feature_order[
            df_feature_order["matrix"] == matrix_name
        ]

        print(f"\n{matrix_name}")

        print(
            "Same feature set:",
            int(subset["same_feature_set"].sum()),
            "/",
            len(subset)
        )

        print(
            "Same feature order:",
            int(subset["same_feature_order"].sum()),
            "/",
            len(subset)
        )


# ============================================================
# 11. FEATURE DIFFERENCES
# ============================================================

print("\n" + "=" * 75)
print("FEATURE DIFFERENCES")
print("=" * 75)

if not df_feature_differences.empty:

    display(
        df_feature_differences[
            (
                (df_feature_differences["missing_feature_count"] > 0)
                | (df_feature_differences["extra_feature_count"] > 0)
            )
        ].head(50)
    )

else:

    print("No feature differences found.")


# ============================================================
# 12. FAILED GRAPH CHECKS
# ============================================================

print("\n" + "=" * 75)
print("FAILED GRAPH CHECKS")
print("=" * 75)

if not df_graph_checks.empty:

    failed = df_graph_checks[
        df_graph_checks["status"] == "FAIL"
    ]

    if not failed.empty:
        display(failed)
    else:
        print("No failed graph checks.")

else:

    print("No graph checks available.")


# ============================================================
# 13. SAVE REPORTS
# ============================================================

if "BATCH_RUN_DIR" not in globals():
    raise RuntimeError("BATCH_RUN_DIR не найден.")

BATCH_RUN_DIR = Path(BATCH_RUN_DIR)
BATCH_RUN_DIR.mkdir(parents=True, exist_ok=True)

reports = {
    "feature_schema_details.csv": df_feature_schemas,
    "feature_schema_variants.csv": df_feature_summary,
    "feature_schema_differences.csv": df_feature_differences,
    "feature_schema_order.csv": df_feature_order,
    "graph_integrity_checks.csv": df_graph_checks
}

for filename, dataframe in reports.items():

    output_path = BATCH_RUN_DIR / filename

    dataframe.to_csv(
        output_path,
        index=False,
        encoding="utf-8-sig"
    )

    print("Saved:", output_path)


# ============================================================
# 14. FINAL STATUS
# ============================================================

print("\n" + "=" * 75)
print("CELL 10 COMPLETED")
print("=" * 75)

print("Structures audited:", len(audit_input))

for matrix_name, schema_dict in schemas_by_type.items():
    print(
        f"{matrix_name}: "
        f"{len(schema_dict)} schema variants"
    )

if not df_graph_checks.empty:
    print(
        "Failed graph checks:",
        int(
            (
                df_graph_checks["status"] == "FAIL"
            ).sum()
        )
    )

print("Report directory:", BATCH_RUN_DIR)
print("Finished:", datetime.now().isoformat(timespec="seconds"))
print("=" * 75)


CELL 10. FEATURE SCHEMA AND GRAPH INTEGRITY AUDIT
Structures: 100
Started: 2026-11-07T15:16:34
[1/100] 7WTU
[2/100] 7W37
[3/100] 7Q0G
[4/100] 7VYS
[5/100] 7X8R
[6/100] 7QB5
[7/100] 7Z0O
[8/100] 7XCR
[9/100] 12TB
[10/100] 3PUW
[11/100] 7WTT
[12/100] 3O2R
[13/100] 3NV0
[14/100] 7Z16
[15/100] 7WSO
[16/100] 7PR4
[17/100] 7W38
[18/100] 3OFG
[19/100] 7XNX
[20/100] 3P1A
[21/100] 7W1Z
[22/100] 7OYK
[23/100] 7WIG
[24/100] 7PS6
[25/100] 7W2U
[26/100] 3NZJ
[27/100] 7XCT
[28/100] 7YMS
[29/100] 3NY7
[30/100] 3P0Y
[31/100] 7Y3G
[32/100] 7YE4
[33/100] 7P4R
[34/100] 7W5Z
[35/100] 7PCA
[36/100] 7WU2
[37/100] 3OS8
[38/100] 7YE6
[39/100] 3P3O
[40/100] 7Z13
[41/100] 7PEF
[42/100] 7XD0
[43/100] 7PU4
[44/100] 3OR7
[45/100] 7W2Y
[46/100] 3O0A
[47/100] 7XMT
[48/100] 3PC8
[49/100] 21ZV
[50/100] 3O1C
[51/100] 7Y7M
[52/100] 3PSN
[53/100] 7W4C
[54/100] 3NWS
[55/100] 7Z38
[56/100] 3POZ
[57/100] 7VZ4
[58/100] 3OZZ
[59/100] 7Y75
[60/100] 3P8V
[61/100] 7XCP
[62/100] 3O1X
[63/100] 7XW6
[64/100] 3NZL
[65/100] 7W4F
[66/

,status,checks
0,PASS,1500
1,FAIL,100


Failed checks: 100
Warnings: 0

FEATURE SCHEMA VARIANTS


,matrix,schema_number,structures_count,feature_count,pdb_ids
0,X_atom,1,45,26,"7WTU, 7Q0G, 7VYS, 7Z0O, 7WSO, 7XNX, 7W1Z, 7WIG..."
1,X_atom,2,52,27,"7W37, 7X8R, 7XCR, 12TB, 3PUW, 7WTT, 3O2R, 3NV0..."
2,X_atom,3,1,28,7QB5
3,X_atom,4,1,28,3NZJ
4,X_atom,5,1,29,7Z13
5,X_residue,1,97,23,"7WTU, 7W37, 7Q0G, 7VYS, 7X8R, 7Z0O, 7XCR, 12TB..."
6,X_residue,2,1,24,7QB5
7,X_residue,3,2,25,"3NZJ, 7Z13"



FEATURE ORDER COMPARISON

X_atom
Same feature set: 45 / 100
Same feature order: 45 / 100

X_residue
Same feature set: 97 / 100
Same feature order: 97 / 100

X_global
Same feature set: 0 / 0
Same feature order: 0 / 0

FEATURE DIFFERENCES


,pdb_id,matrix,feature_count,missing_feature_count,extra_feature_count,missing_features,extra_features
0,7WTU,X_atom,26,3,0,numeric_chain_id_display | numeric_chain_id_di...,
1,7W37,X_atom,27,2,0,numeric_chain_id_display | numeric_chain_id_di...,
2,7Q0G,X_atom,26,3,0,numeric_chain_id_display | numeric_chain_id_di...,
3,7VYS,X_atom,26,3,0,numeric_chain_id_display | numeric_chain_id_di...,
4,7X8R,X_atom,27,2,0,numeric_chain_id_display | numeric_chain_id_di...,
5,7QB5,X_atom,28,1,0,numeric_chain_id_display__missing,
6,7Z0O,X_atom,26,3,0,numeric_chain_id_display | numeric_chain_id_di...,
7,7XCR,X_atom,27,2,0,numeric_chain_id_display | numeric_chain_id_di...,
8,12TB,X_atom,27,2,0,numeric_chain_id_display | numeric_chain_id_di...,
9,3PUW,X_atom,27,2,0,numeric_chain_id_display | numeric_chain_id_di...,



FAILED GRAPH CHECKS


,pdb_id,check,status,details
4,7WTU,X_global_NDIM,FAIL,"Expected 2D matrix, got (126,)"
20,7W37,X_global_NDIM,FAIL,"Expected 2D matrix, got (127,)"
36,7Q0G,X_global_NDIM,FAIL,"Expected 2D matrix, got (124,)"
52,7VYS,X_global_NDIM,FAIL,"Expected 2D matrix, got (122,)"
68,7X8R,X_global_NDIM,FAIL,"Expected 2D matrix, got (117,)"
...,...,...,...,...
1524,7WU3,X_global_NDIM,FAIL,"Expected 2D matrix, got (117,)"
1540,3PAM,X_global_NDIM,FAIL,"Expected 2D matrix, got (118,)"
1556,7XD1,X_global_NDIM,FAIL,"Expected 2D matrix, got (119,)"
1572,7P0J,X_global_NDIM,FAIL,"Expected 2D matrix, got (121,)"


Saved: C:\apex\APEX\МИФИ\СНО\medic\Protein_Pocket_Dataset\07_structure_analysis\batch_runs\protein_batch_v1\feature_schema_details.csv
Saved: C:\apex\APEX\МИФИ\СНО\medic\Protein_Pocket_Dataset\07_structure_analysis\batch_runs\protein_batch_v1\feature_schema_variants.csv
Saved: C:\apex\APEX\МИФИ\СНО\medic\Protein_Pocket_Dataset\07_structure_analysis\batch_runs\protein_batch_v1\feature_schema_differences.csv
Saved: C:\apex\APEX\МИФИ\СНО\medic\Protein_Pocket_Dataset\07_structure_analysis\batch_runs\protein_batch_v1\feature_schema_order.csv
Saved: C:\apex\APEX\МИФИ\СНО\medic\Protein_Pocket_Dataset\07_structure_analysis\batch_runs\protein_batch_v1\graph_integrity_checks.csv

CELL 10 COMPLETED
Structures audited: 100
X_atom: 5 schema variants
X_residue: 3 schema variants
X_global: 0 schema variants
Failed graph checks: 100
Report directory: C:\apex\APEX\МИФИ\СНО\medic\Protein_Pocket_Dataset\07_structure_analysis\batch_runs\protein_batch_v1
Finished: 2026-11-07T15:16:38


In [13]:
import os

ROOT_DIR = r"C:\apex\APEX\МИФИ\СНО\medic\Protein_Pocket_Dataset\07_structure_analysis\results"

OUTPUT_FILE = os.path.join(
    ROOT_DIR,
    "results_directory_tree.txt"
)

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:

    for root, dirs, files in os.walk(ROOT_DIR):

        # Сортировка директорий и файлов
        dirs.sort()
        files.sort()

        level = root.replace(ROOT_DIR, "").count(os.sep)
        indent = "    " * level

        line = f"{indent}{os.path.basename(root)}/"

        print(line)
        f.write(line + "\n")

        for file in files:

            # Не включаем сам файл отчёта в список
            if file == os.path.basename(OUTPUT_FILE):
                continue

            line = f"{indent}    {file}"

            print(line)
            f.write(line + "\n")

print("\nСтруктура каталогов сохранена:")
print(OUTPUT_FILE)

results/
    11PH/
        11PH_all_final_metrics.csv
        11PH_altloc_atoms.csv
        11PH_amino_acid_classification.csv
        11PH_amino_acid_distribution.csv
        11PH_amino_acid_distribution.png
        11PH_amino_acid_distribution_by_chain.csv
        11PH_amino_acid_percentage.png
        11PH_amino_acid_summary.json
        11PH_analysis_report.txt
        11PH_atom_analysis_summary.json
        11PH_atom_quality.csv
        11PH_binding_atom_contacts.csv
        11PH_binding_site.csv
        11PH_binding_site.json
        11PH_binding_site_summary.csv
        11PH_cell4_verification.csv
        11PH_chain_atom_statistics.csv
        11PH_chain_entity_mapping.csv
        11PH_chain_identity.csv
        11PH_chain_statistics.csv
        11PH_chemical_component_dictionary.csv
        11PH_component_atoms.csv
        11PH_component_distribution.csv
        11PH_component_info.json
        11PH_component_occurrences.csv
        11PH_component_separation.json
        11PH_c

In [14]:

# ============================================================
# AUDIT 1. STRUCTURE OF RESULTS FOR ALL PROTEINS
# ============================================================

from pathlib import Path
from collections import Counter, defaultdict
import pandas as pd
import csv

ROOT_DIR = Path(
    r"C:\apex\APEX\МИФИ\СНО\medic\Protein_Pocket_Dataset"
    r"\07_structure_analysis\results"
)

AUDIT_DIR = ROOT_DIR / "audit"
AUDIT_DIR.mkdir(exist_ok=True)

# ------------------------------------------------------------
# 1. Find protein directories
# ------------------------------------------------------------

protein_dirs = sorted(
    p for p in ROOT_DIR.iterdir()
    if p.is_dir() and p.name != "audit"
)

print("=" * 70)
print("AUDIT OF PROTEIN RESULTS")
print("=" * 70)

print(f"Protein directories: {len(protein_dirs)}")

# ------------------------------------------------------------
# 2. Collect file information
# ------------------------------------------------------------

file_records = []
file_frequency = Counter()
csv_schemas = defaultdict(list)
errors = []

for protein_dir in protein_dirs:

    files = sorted(
        p for p in protein_dir.iterdir()
        if p.is_file()
    )

    for file in files:

        file_frequency[file.name] += 1

        record = {
            "pdb_id": protein_dir.name,
            "file_name": file.name,
            "extension": file.suffix.lower(),
            "size_bytes": file.stat().st_size,
            "read_status": "OK",
            "columns": None
        }

        # ----------------------------------------------------
        # 3. Read CSV headers only
        # ----------------------------------------------------

        if file.suffix.lower() == ".csv":

            try:
                with open(
                    file,
                    "r",
                    encoding="utf-8-sig",
                    newline=""
                ) as f:

                    reader = csv.reader(f)
                    columns = next(reader, [])

                record["columns"] = " | ".join(columns)

                schema_key = (
                    file.name,
                    tuple(columns)
                )

                csv_schemas[schema_key].append(
                    protein_dir.name
                )

            except Exception as e:

                record["read_status"] = "ERROR"
                errors.append({
                    "pdb_id": protein_dir.name,
                    "file": file.name,
                    "error": str(e)
                })

        file_records.append(record)

# ------------------------------------------------------------
# 4. Save detailed file inventory
# ------------------------------------------------------------

df_files = pd.DataFrame(file_records)

df_files.to_csv(
    AUDIT_DIR / "audit_file_inventory.csv",
    index=False,
    encoding="utf-8-sig"
)

# ------------------------------------------------------------
# 5. File frequency report
# ------------------------------------------------------------

df_frequency = pd.DataFrame(
    [
        {
            "file_name": name,
            "protein_count": count,
            "missing_count": len(protein_dirs) - count
        }
        for name, count in file_frequency.items()
    ]
).sort_values("file_name")

df_frequency.to_csv(
    AUDIT_DIR / "audit_file_frequency.csv",
    index=False,
    encoding="utf-8-sig"
)

# ------------------------------------------------------------
# 6. CSV schema comparison
# ------------------------------------------------------------

schema_records = []

for (file_name, columns), pdb_ids in csv_schemas.items():

    schema_records.append({
        "file_name": file_name,
        "schema_count": len(columns),
        "protein_count": len(pdb_ids),
        "pdb_ids": ", ".join(pdb_ids),
        "columns": " | ".join(columns)
    })

df_schemas = pd.DataFrame(schema_records)

df_schemas.to_csv(
    AUDIT_DIR / "audit_csv_schemas.csv",
    index=False,
    encoding="utf-8-sig"
)

# ------------------------------------------------------------
# 7. Read errors
# ------------------------------------------------------------

df_errors = pd.DataFrame(errors)

df_errors.to_csv(
    AUDIT_DIR / "audit_errors.csv",
    index=False,
    encoding="utf-8-sig"
)

# ------------------------------------------------------------
# 8. Console summary
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("SUMMARY")
print("=" * 70)

print(f"Protein directories: {len(protein_dirs)}")
print(f"Total files: {len(file_records)}")
print(f"Unique file names: {len(file_frequency)}")
print(f"CSV files: {sum(1 for x in file_records if x['extension'] == '.csv')}")
print(f"CSV reading errors: {len(errors)}")

print("\nFILE FREQUENCY:")
print(df_frequency.to_string(index=False))

print("\nCSV SCHEMAS:")
print(
    df_schemas[
        ["file_name", "schema_count", "protein_count"]
    ].to_string(index=False)
)

print("\nAUDIT REPORTS:")
print(AUDIT_DIR)

print("\nAUDIT COMPLETED")


AUDIT OF PROTEIN RESULTS
Protein directories: 100

SUMMARY
Protein directories: 100
Total files: 10709
Unique file names: 9719
CSV files: 8409
CSV reading errors: 0

FILE FREQUENCY:
                                    file_name  protein_count  missing_count
                   11PH_all_final_metrics.csv              1             99
                        11PH_altloc_atoms.csv              1             99
           11PH_amino_acid_classification.csv              1             99
             11PH_amino_acid_distribution.csv              1             99
             11PH_amino_acid_distribution.png              1             99
    11PH_amino_acid_distribution_by_chain.csv              1             99
               11PH_amino_acid_percentage.png              1             99
                 11PH_amino_acid_summary.json              1             99
                     11PH_analysis_report.txt              1             99
              11PH_atom_analysis_summary.json             

In [16]:

# ============================================================
# AUDIT 2. NORMALIZED FILES AND CSV SCHEMAS
# ============================================================

from pathlib import Path
import pandas as pd

ROOT_DIR = Path(
    r"C:\apex\APEX\МИФИ\СНО\medic\Protein_Pocket_Dataset"
    r"\07_structure_analysis\results"
)

AUDIT_DIR = ROOT_DIR / "audit"
INVENTORY_FILE = AUDIT_DIR / "audit_file_inventory.csv"

# ------------------------------------------------------------
# 1. Load inventory
# ------------------------------------------------------------

df = pd.read_csv(
    INVENTORY_FILE,
    encoding="utf-8-sig",
    dtype={"pdb_id": str, "file_name": str}
)

protein_dirs = sorted(
    p.name for p in ROOT_DIR.iterdir()
    if p.is_dir() and p.name != "audit"
)

protein_count = len(protein_dirs)

# ------------------------------------------------------------
# 2. Normalize file names
# ------------------------------------------------------------

def normalize_filename(row):
    prefix = str(row["pdb_id"]) + "_"
    filename = str(row["file_name"])

    if filename.startswith(prefix):
        return filename[len(prefix):]

    return filename


df["normalized_file_name"] = df.apply(
    normalize_filename,
    axis=1
)

# ------------------------------------------------------------
# 3. File frequency by normalized name
# ------------------------------------------------------------

df_frequency = (
    df.groupby("normalized_file_name")
    .agg(
        protein_count=("pdb_id", "nunique"),
        total_file_count=("file_name", "count")
    )
    .reset_index()
)

df_frequency["missing_count"] = (
    protein_count - df_frequency["protein_count"]
)

df_frequency = df_frequency.sort_values(
    ["missing_count", "normalized_file_name"]
)

df_frequency.to_csv(
    AUDIT_DIR / "audit2_normalized_file_frequency.csv",
    index=False,
    encoding="utf-8-sig"
)

# ------------------------------------------------------------
# 4. Missing files by protein
# ------------------------------------------------------------

all_normalized_files = sorted(
    df["normalized_file_name"].dropna().unique()
)

present = set(
    zip(df["pdb_id"], df["normalized_file_name"])
)

missing_records = []

for pdb_id in protein_dirs:
    for filename in all_normalized_files:
        if (pdb_id, filename) not in present:
            missing_records.append({
                "pdb_id": pdb_id,
                "missing_file": filename
            })

df_missing = pd.DataFrame(missing_records)

df_missing.to_csv(
    AUDIT_DIR / "audit2_missing_files.csv",
    index=False,
    encoding="utf-8-sig"
)

# ------------------------------------------------------------
# 5. Compare CSV schemas by normalized name
# ------------------------------------------------------------

df_csv = df[
    df["extension"].str.lower() == ".csv"
].copy()

schema_groups = (
    df_csv.groupby(
        ["normalized_file_name", "columns"],
        dropna=False
    )
    .agg(
        protein_count=("pdb_id", "nunique"),
        pdb_ids=("pdb_id", lambda x: ", ".join(sorted(set(x))))
    )
    .reset_index()
)

schema_groups.to_csv(
    AUDIT_DIR / "audit2_normalized_csv_schemas.csv",
    index=False,
    encoding="utf-8-sig"
)

# ------------------------------------------------------------
# 6. Identify files with different schemas
# ------------------------------------------------------------

schema_variants = (
    schema_groups.groupby("normalized_file_name")
    .agg(
        schema_variants=("columns", "nunique"),
        total_proteins=("protein_count", "sum")
    )
    .reset_index()
)

df_schema_differences = schema_variants[
    schema_variants["schema_variants"] > 1
].copy()

df_schema_differences.to_csv(
    AUDIT_DIR / "audit2_schema_differences.csv",
    index=False,
    encoding="utf-8-sig"
)

# ------------------------------------------------------------
# 7. Duplicate normalized files
# ------------------------------------------------------------

df_duplicates = (
    df.groupby(["pdb_id", "normalized_file_name"])
    .size()
    .reset_index(name="count")
)

df_duplicates = df_duplicates[
    df_duplicates["count"] > 1
]

df_duplicates.to_csv(
    AUDIT_DIR / "audit2_duplicates.csv",
    index=False,
    encoding="utf-8-sig"
)

# ------------------------------------------------------------
# 8. Summary
# ------------------------------------------------------------

print("=" * 70)
print("NORMALIZED AUDIT SUMMARY")
print("=" * 70)

print("Protein directories:", protein_count)
print("Inventory records:", len(df))
print("Unique normalized file names:", len(all_normalized_files))
print("Unique normalized CSV names:",
      df_csv["normalized_file_name"].nunique())

print("Missing file records:", len(df_missing))
print("CSV files with different schemas:",
      len(df_schema_differences))
print("Duplicate normalized files:", len(df_duplicates))

print("\nMOST COMMON FILES:")
print(
    df_frequency.sort_values(
        "protein_count",
        ascending=False
    ).head(30).to_string(index=False)
)

print("\nSCHEMA DIFFERENCES:")
if len(df_schema_differences):
    print(df_schema_differences.to_string(index=False))
else:
    print("No schema differences detected.")

print("\nREPORT DIRECTORY:")
print(AUDIT_DIR)

print("\nAUDIT 2 COMPLETED")


NORMALIZED AUDIT SUMMARY
Protein directories: 100
Inventory records: 10709
Unique normalized file names: 110
Unique normalized CSV names: 87
Missing file records: 291
CSV files with different schemas: 2
Duplicate normalized files: 0

MOST COMMON FILES:
                normalized_file_name  protein_count  total_file_count  missing_count
               all_final_metrics.csv            100               100              0
                    altloc_atoms.csv            100               100              0
       amino_acid_classification.csv            100               100              0
         amino_acid_distribution.csv            100               100              0
         amino_acid_distribution.png            100               100              0
amino_acid_distribution_by_chain.csv            100               100              0
           amino_acid_percentage.png            100               100              0
             amino_acid_summary.json            100               1

In [17]:

# ============================================================
# CELL AUDIT 3. ALL_FINAL_METRICS AUDIT
# ============================================================
# Назначение:
# 1. Проверить all_final_metrics.csv во всех папках PDB.
# 2. Проверить наличие и читаемость файлов.
# 3. Сравнить структуру CSV и названия столбцов.
# 4. Проверить покрытие метрик по всем белкам.
# 5. Найти дубли метрик внутри одного PDB.
# 6. Найти конфликты единиц измерения.
# 7. Проверить пропуски, числовые и нечисловые значения.
# 8. Проверить распределение статусов.
#
# ВАЖНО:
# - Исходные файлы не изменяются.
# - Все отчёты сохраняются в results/audit/.
# - Нечисловые значения не удаляются и не исправляются.
# ============================================================

from pathlib import Path
from collections import Counter
import pandas as pd
import numpy as np
import re
import traceback

# ------------------------------------------------------------
# 1. PATHS AND SETTINGS
# ------------------------------------------------------------

PROJECT_DIR = Path(
    r"C:\apex\APEX\МИФИ\СНО\medic\Protein_Pocket_Dataset\07_structure_analysis"
)

RESULTS_DIR = PROJECT_DIR / "results"
AUDIT_DIR = RESULTS_DIR / "audit"

AUDIT_DIR.mkdir(parents=True, exist_ok=True)

REQUIRED_COLUMNS = [
    "category",
    "metric",
    "value",
    "unit",
    "status"
]

print("=" * 70)
print("CELL AUDIT 3. ALL_FINAL_METRICS AUDIT")
print("=" * 70)

print(f"Results directory: {RESULTS_DIR}")
print(f"Audit directory:   {AUDIT_DIR}")

if not RESULTS_DIR.exists():
    raise FileNotFoundError(
        f"Results directory not found: {RESULTS_DIR}"
    )

# ------------------------------------------------------------
# 2. FIND PROTEIN DIRECTORIES
# ------------------------------------------------------------

protein_dirs = sorted([
    path
    for path in RESULTS_DIR.iterdir()
    if path.is_dir() and path.name.lower() != "audit"
])

print(f"\nProtein directories found: {len(protein_dirs)}")

# ------------------------------------------------------------
# 3. FIND all_final_metrics.csv
# ------------------------------------------------------------

def find_metrics_file(protein_dir):
    """
    Ищет файл all_final_metrics.csv.

    Приоритет:
    1. <PDB_ID>_all_final_metrics.csv
    2. all_final_metrics.csv
    3. Другие файлы с окончанием _all_final_metrics.csv
    """

    pdb_id = protein_dir.name

    candidates = [
        protein_dir / f"{pdb_id}_all_final_metrics.csv",
        protein_dir / "all_final_metrics.csv"
    ]

    for candidate in candidates:
        if candidate.is_file():
            return candidate, "FOUND"

    fallback = sorted(
        protein_dir.glob("*_all_final_metrics.csv")
    )

    if len(fallback) == 1:
        return fallback[0], "FOUND_FALLBACK"

    if len(fallback) > 1:
        return None, "AMBIGUOUS_MULTIPLE_FILES"

    return None, "FILE_NOT_FOUND"


# ------------------------------------------------------------
# 4. READ ALL FILES
# ------------------------------------------------------------

all_data = []

inventory_records = []
schema_records = []
error_records = []

for i, protein_dir in enumerate(protein_dirs, start=1):

    pdb_id = protein_dir.name

    file_path, search_status = find_metrics_file(protein_dir)

    if file_path is None:

        inventory_records.append({
            "pdb_id": pdb_id,
            "file": None,
            "search_status": search_status,
            "read_status": "NOT_READ",
            "rows": 0,
            "columns_count": 0,
            "columns": None
        })

        if search_status != "FILE_NOT_FOUND":
            error_records.append({
                "pdb_id": pdb_id,
                "file": None,
                "error_type": search_status,
                "error_message": "Multiple candidate files found"
            })

        continue

    try:

        df = pd.read_csv(
            file_path,
            encoding="utf-8-sig"
        )

        # Убираем случайные пробелы вокруг названий столбцов.
        # Данные в самих столбцах не изменяем.
        df.columns = [
            str(col).strip()
            for col in df.columns
        ]

        columns = list(df.columns)

        missing_columns = [
            col
            for col in REQUIRED_COLUMNS
            if col not in columns
        ]

        schema_records.append({
            "pdb_id": pdb_id,
            "file": file_path.name,
            "columns_count": len(columns),
            "columns": " | ".join(columns),
            "missing_required_columns": " | ".join(missing_columns)
        })

        inventory_records.append({
            "pdb_id": pdb_id,
            "file": file_path.name,
            "search_status": search_status,
            "read_status": "OK",
            "rows": len(df),
            "columns_count": len(columns),
            "columns": " | ".join(columns)
        })

        # Если обязательные поля отсутствуют,
        # сохраняем информацию, но не используем файл
        # для анализа метрик.
        if missing_columns:

            error_records.append({
                "pdb_id": pdb_id,
                "file": file_path.name,
                "error_type": "MISSING_REQUIRED_COLUMNS",
                "error_message": ", ".join(missing_columns)
            })

            continue

        # Добавляем служебные поля только в оперативную копию.
        df = df.copy()

        df["_pdb_id"] = pdb_id
        df["_source_file"] = file_path.name

        all_data.append(df)

    except Exception as exc:

        inventory_records.append({
            "pdb_id": pdb_id,
            "file": file_path.name,
            "search_status": search_status,
            "read_status": "READ_ERROR",
            "rows": 0,
            "columns_count": 0,
            "columns": None
        })

        error_records.append({
            "pdb_id": pdb_id,
            "file": file_path.name,
            "error_type": type(exc).__name__,
            "error_message": str(exc)
        })


# ------------------------------------------------------------
# 5. BUILD INVENTORY AND COMBINED DATASET
# ------------------------------------------------------------

inventory_df = pd.DataFrame(inventory_records)
schema_df = pd.DataFrame(schema_records)
errors_df = pd.DataFrame(error_records)

if all_data:

    metrics_df = pd.concat(
        all_data,
        ignore_index=True,
        sort=False
    )

else:

    metrics_df = pd.DataFrame()

print("\n" + "-" * 70)
print("FILE PROCESSING SUMMARY")
print("-" * 70)

if not inventory_df.empty:

    print(
        "Directories checked:",
        len(inventory_df)
    )

    print(
        "Files read successfully:",
        int((inventory_df["read_status"] == "OK").sum())
    )

    print(
        "Files not found:",
        int((inventory_df["search_status"] == "FILE_NOT_FOUND").sum())
    )

    print(
        "Read errors:",
        int((inventory_df["read_status"] == "READ_ERROR").sum())
    )

    print(
        "Ambiguous files:",
        int((inventory_df["search_status"] == "AMBIGUOUS_MULTIPLE_FILES").sum())
    )

print(
    "Total metric rows:",
    len(metrics_df)
)


# ------------------------------------------------------------
# 6. CHECK CSV SCHEMA CONSISTENCY
# ------------------------------------------------------------

schema_difference_records = []

if not schema_df.empty:

    schema_counter = Counter(
        tuple(str(col).strip().split(" | "))
        for col in schema_df["columns"]
    )

    print("\nSchema variants found:", len(schema_counter))

    for schema_number, (columns, count) in enumerate(
        schema_counter.items(),
        start=1
    ):

        schema_difference_records.append({
            "schema_id": schema_number,
            "files_count": count,
            "columns_count": len(columns),
            "columns": " | ".join(columns)
        })

schema_variants_df = pd.DataFrame(
    schema_difference_records
)


# ------------------------------------------------------------
# 7. METRIC DATA QUALITY
# ------------------------------------------------------------

coverage_df = pd.DataFrame()
status_df = pd.DataFrame()
duplicates_df = pd.DataFrame()
unit_conflicts_df = pd.DataFrame()
value_quality_df = pd.DataFrame()
missing_key_df = pd.DataFrame()

if not metrics_df.empty:

    # Нормализуем только временные поля для группировки.
    # Исходные значения остаются в metrics_df.

    for col in ["category", "metric", "unit", "status"]:

        metrics_df[f"_{col}_key"] = (
            metrics_df[col]
            .astype("string")
            .str.strip()
            .replace("", pd.NA)
        )

    # --------------------------------------------------------
    # 7.1. Missing category / metric
    # --------------------------------------------------------

    missing_key_df = metrics_df[
        metrics_df["_category_key"].isna()
        | metrics_df["_metric_key"].isna()
    ].copy()

    # --------------------------------------------------------
    # 7.2. Value parsing
    # --------------------------------------------------------

    raw_value = metrics_df["value"]

    value_as_text = (
        raw_value
        .astype("string")
        .str.strip()
    )

    empty_value_mask = (
        raw_value.isna()
        | value_as_text.eq("")
    )

    numeric_value = pd.to_numeric(
        raw_value,
        errors="coerce"
    )

    non_numeric_mask = (
        ~empty_value_mask
        & numeric_value.isna()
    )

    metrics_df["_value_numeric"] = numeric_value
    metrics_df["_value_missing"] = empty_value_mask
    metrics_df["_value_non_numeric"] = non_numeric_mask

    # --------------------------------------------------------
    # 7.3. Metric coverage
    # --------------------------------------------------------

    valid_metric_rows = metrics_df[
        metrics_df["_category_key"].notna()
        & metrics_df["_metric_key"].notna()
    ].copy()

    group_columns = [
        "_category_key",
        "_metric_key",
        "_unit_key"
    ]

    if not valid_metric_rows.empty:

        coverage_df = (
            valid_metric_rows
            .groupby(
                group_columns,
                dropna=False
            )
            .agg(
                rows_count=("_pdb_id", "size"),
                proteins_count=("_pdb_id", "nunique"),
                missing_values_count=("_value_missing", "sum"),
                numeric_values_count=("_value_numeric", "count"),
                non_numeric_values_count=("_value_non_numeric", "sum"),
                statuses_count=("_status_key", "nunique")
            )
            .reset_index()
        )

        coverage_df = coverage_df.rename(columns={
            "_category_key": "category",
            "_metric_key": "metric",
            "_unit_key": "unit"
        })

        coverage_df["coverage_percent"] = (
            coverage_df["proteins_count"]
            / max(len(protein_dirs), 1)
            * 100
        )

        coverage_df = coverage_df.sort_values(
            ["category", "metric", "unit"],
            na_position="last"
        )

    # --------------------------------------------------------
    # 7.4. Status distribution per metric
    # --------------------------------------------------------

    status_data = valid_metric_rows[
        valid_metric_rows["_status_key"].notna()
    ]

    if not status_data.empty:

        status_df = (
            status_data
            .groupby(
                [
                    "_category_key",
                    "_metric_key",
                    "_unit_key",
                    "_status_key"
                ],
                dropna=False
            )
            .agg(
                rows_count=("_pdb_id", "size"),
                proteins_count=("_pdb_id", "nunique")
            )
            .reset_index()
            .rename(columns={
                "_category_key": "category",
                "_metric_key": "metric",
                "_unit_key": "unit",
                "_status_key": "status"
            })
        )

    # --------------------------------------------------------
    # 7.5. Duplicate metrics within one protein
    # --------------------------------------------------------

    duplicate_key = [
        "_pdb_id",
        "_category_key",
        "_metric_key",
        "_unit_key"
    ]

    duplicate_mask = valid_metric_rows.duplicated(
        subset=duplicate_key,
        keep=False
    )

    duplicates_df = valid_metric_rows[
        duplicate_mask
    ].copy()

    if not duplicates_df.empty:

        duplicates_df = duplicates_df.sort_values(
            duplicate_key
        )

        duplicates_df = duplicates_df.rename(columns={
            "_pdb_id": "pdb_id",
            "_category_key": "category",
            "_metric_key": "metric",
            "_unit_key": "unit"
        })

    # --------------------------------------------------------
    # 7.6. Unit conflicts for the same category + metric
    # --------------------------------------------------------

    unit_data = valid_metric_rows[
        valid_metric_rows["_unit_key"].notna()
    ]

    if not unit_data.empty:

        unit_counts = (
            unit_data
            .groupby(
                ["_category_key", "_metric_key"],
                dropna=False
            )["_unit_key"]
            .nunique()
            .reset_index(name="different_units_count")
        )

        conflicting_keys = unit_counts[
            unit_counts["different_units_count"] > 1
        ]

        if not conflicting_keys.empty:

            unit_conflicts_df = unit_data.merge(
                conflicting_keys,
                on=["_category_key", "_metric_key"],
                how="inner"
            )

            unit_conflicts_df = (
                unit_conflicts_df
                .groupby(
                    [
                        "_category_key",
                        "_metric_key",
                        "_unit_key"
                    ],
                    dropna=False
                )
                .agg(
                    proteins_count=("_pdb_id", "nunique"),
                    rows_count=("_pdb_id", "size")
                )
                .reset_index()
                .rename(columns={
                    "_category_key": "category",
                    "_metric_key": "metric",
                    "_unit_key": "unit"
                })
            )

    # --------------------------------------------------------
    # 7.7. Numeric value quality
    # --------------------------------------------------------

    value_quality_df = (
        valid_metric_rows
        .groupby(
            group_columns,
            dropna=False
        )
        .agg(
            rows_count=("_pdb_id", "size"),
            numeric_count=("_value_numeric", "count"),
            missing_count=("_value_missing", "sum"),
            non_numeric_count=("_value_non_numeric", "sum"),
            numeric_min=("_value_numeric", "min"),
            numeric_max=("_value_numeric", "max"),
            numeric_mean=("_value_numeric", "mean")
        )
        .reset_index()
        .rename(columns={
            "_category_key": "category",
            "_metric_key": "metric",
            "_unit_key": "unit"
        })
    )


# ------------------------------------------------------------
# 8. SAVE REPORTS
# ------------------------------------------------------------

reports = {
    "audit3_all_final_metrics_inventory.csv": inventory_df,
    "audit3_all_final_metrics_schemas.csv": schema_df,
    "audit3_schema_variants.csv": schema_variants_df,
    "audit3_metric_coverage.csv": coverage_df,
    "audit3_metric_status.csv": status_df,
    "audit3_duplicate_metrics.csv": duplicates_df,
    "audit3_unit_conflicts.csv": unit_conflicts_df,
    "audit3_metric_value_quality.csv": value_quality_df,
    "audit3_missing_metric_keys.csv": missing_key_df,
    "audit3_errors.csv": errors_df
}

for filename, report_df in reports.items():

    report_path = AUDIT_DIR / filename

    report_df.to_csv(
        report_path,
        index=False,
        encoding="utf-8-sig"
    )


# ------------------------------------------------------------
# 9. SUMMARY
# ------------------------------------------------------------

files_found = (
    int((inventory_df["read_status"] == "OK").sum())
    if not inventory_df.empty else 0
)

files_missing = (
    int((inventory_df["search_status"] == "FILE_NOT_FOUND").sum())
    if not inventory_df.empty else 0
)

files_with_errors = (
    int((inventory_df["read_status"] == "READ_ERROR").sum())
    if not inventory_df.empty else 0
)

schema_variant_count = len(schema_variants_df)

duplicate_group_count = 0

if not duplicates_df.empty:

    duplicate_group_count = (
        duplicates_df
        .groupby(duplicate_key, dropna=False)
        .ngroups
    )

unit_conflict_count = (
    unit_conflicts_df[
        ["category", "metric"]
    ].drop_duplicates().shape[0]
    if not unit_conflicts_df.empty else 0
)

metrics_count = (
    coverage_df[
        ["category", "metric"]
    ].drop_duplicates().shape[0]
    if not coverage_df.empty else 0
)

summary_lines = [
    "CELL AUDIT 3 SUMMARY",
    "=" * 60,
    f"Protein directories: {len(protein_dirs)}",
    f"Files read successfully: {files_found}",
    f"Files not found: {files_missing}",
    f"Files with read errors: {files_with_errors}",
    f"Total metric rows: {len(metrics_df)}",
    f"Distinct category/metric pairs: {metrics_count}",
    f"Schema variants: {schema_variant_count}",
    f"Duplicate metric groups: {duplicate_group_count}",
    f"Metrics with unit conflicts: {unit_conflict_count}",
    f"Rows with missing category/metric: {len(missing_key_df)}",
    "",
    "Reports saved to:",
    str(AUDIT_DIR)
]

summary_text = "\n".join(summary_lines)

summary_path = AUDIT_DIR / "audit3_summary.txt"

summary_path.write_text(
    summary_text,
    encoding="utf-8"
)

print("\n" + "=" * 70)
print(summary_text)
print("=" * 70)

# ------------------------------------------------------------
# 10. SHOW IMPORTANT RESULTS
# ------------------------------------------------------------

print("\nREPORT FILES:")

for filename in reports:
    print(" -", AUDIT_DIR / filename)

print(" -", summary_path)

if not coverage_df.empty:

    print("\nTOP METRICS BY PROTEIN COVERAGE:")

    display(
        coverage_df
        .sort_values(
            ["proteins_count", "metric"],
            ascending=[False, True]
        )
        .head(30)
    )

if not duplicates_df.empty:

    print("\nDUPLICATE METRICS FOUND:")

    display(duplicates_df.head(30))

else:

    print("\nNo duplicate metrics found.")

if not unit_conflicts_df.empty:

    print("\nUNIT CONFLICTS FOUND:")

    display(unit_conflicts_df)

else:

    print("\nNo unit conflicts found.")

if not errors_df.empty:

    print("\nFILE ERRORS / WARNINGS:")

    display(errors_df)

print("\nCELL AUDIT 3 COMPLETED.")


CELL AUDIT 3. ALL_FINAL_METRICS AUDIT
Results directory: C:\apex\APEX\МИФИ\СНО\medic\Protein_Pocket_Dataset\07_structure_analysis\results
Audit directory:   C:\apex\APEX\МИФИ\СНО\medic\Protein_Pocket_Dataset\07_structure_analysis\results\audit

Protein directories found: 100

----------------------------------------------------------------------
FILE PROCESSING SUMMARY
----------------------------------------------------------------------
Directories checked: 100
Files read successfully: 100
Files not found: 0
Read errors: 0
Ambiguous files: 0
Total metric rows: 36511

Schema variants found: 1

CELL AUDIT 3 SUMMARY
Protein directories: 100
Files read successfully: 100
Files not found: 0
Files with read errors: 0
Total metric rows: 36511
Distinct category/metric pairs: 12599
Schema variants: 1
Duplicate metric groups: 0
Metrics with unit conflicts: 0
Rows with missing category/metric: 0

Reports saved to:
C:\apex\APEX\МИФИ\СНО\medic\Protein_Pocket_Dataset\07_structure_analysis\results\a

,category,metric,unit,rows_count,proteins_count,missing_values_count,numeric_values_count,non_numeric_values_count,statuses_count,coverage_percent
12588,Structure quality,Alternate location atom records,records,100,100,0,100,0,2,100.0
12589,Structure quality,Atom quality records,records,100,100,0,100,0,2,100.0
0,Binding site,Binding site residues,residues,100,100,97,3,0,3,100.0
12590,Structure quality,Invalid occupancy atom records,records,100,100,100,0,0,1,100.0
12591,Structure quality,Low occupancy atom records,records,100,100,46,54,0,2,100.0
12592,Structure quality,Mean B-factor,A^2,100,100,0,100,0,2,100.0
1,Binding site,Mean minimum residue distance,A,100,100,97,3,0,3,100.0
12593,Structure quality,Mean occupancy,<NA>,100,100,0,100,0,2,100.0
12594,Structure quality,Median B-factor,A^2,100,100,0,100,0,2,100.0
2,Binding site,Minimum component-protein distance,A,100,100,97,3,0,3,100.0



No duplicate metrics found.

No unit conflicts found.

CELL AUDIT 3 COMPLETED.
